In [29]:
import os, json, random, math, time, csv
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm import tqdm
# ============================ USER PATHS ============================
IMG_DIR   = r"/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/Kvasir-SEG/images"
MASK_DIR  = r"/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/Kvasir-SEG/masks"
SAVE_PATH = r"/mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/SC/tr-elm/micromodel_v3_compressed_train.keras"  # must end with .keras

USE_MIXED_PRECISION = False
FAST_STEM  = True
IMG_SIZE   = 160                      # [COMPRESS] resolution compression (256 -> 160); must be divisible by 32
CHANNELS   = (32, 64, 96, 128, 192)
EPOCHS     = 100                      # [COMPRESS] fewer epochs; the faster pipeline converges quickly
BATCH_SIZE = 8                        # increased from 4
MAX_LR     = 3e-3
WEIGHT_DECAY = 1e-4
PATIENCE   = 20
SEED       = 42
N_XAI_IMAGES = 8
USE_CACHE  = True                     # [COMPRESS] reuse compressed .npz cache between runs

TARGETS = dict(gmacs=1.0, params_m=0.5, size_mb=2.0, metric=0.95, fps=50.0)

assert SAVE_PATH.endswith(".keras"), "SAVE_PATH must end with .keras"
assert IMG_SIZE % (32 if FAST_STEM else 16) == 0, "IMG_SIZE must be divisible by 32 (FAST_STEM) or 16"
BASE = SAVE_PATH[:-len(".keras")]
STEM_STRIDE = 2 if FAST_STEM else 1
UP_MODE = "nearest" if USE_MIXED_PRECISION else "bilinear"
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
AUTOTUNE = tf.data.AUTOTUNE
METRIC_KEYS = ["dice", "iou", "precision", "recall", "specificity", "accuracy"]
def seed_everything(s):
    random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)


# ------------------------------- MODEL -------------------------------
def dsconv(x, cout, name, last=False):
    x = layers.SeparableConv2D(cout, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.ReLU(name=f"{name}_out" if last else None)(x)


def block(x, cout, name):
    x = dsconv(x, cout, name)
    return dsconv(x, cout, name, last=True)

def build_model(ch=CHANNELS, size=IMG_SIZE):
    c0, c1, c2, c3, c4 = ch
    inp = layers.Input((size, size, 3))
    x = layers.Conv2D(c0, 3, strides=STEM_STRIDE, padding="same", use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU(name="stem_out")(x)

    s0 = block(x, c0, "enc1")
    s1 = block(layers.MaxPool2D()(s0), c1, "enc2")
    s2 = block(layers.MaxPool2D()(s1), c2, "enc3")
    s3 = block(layers.MaxPool2D()(s2), c3, "enc4")
    b  = block(layers.MaxPool2D()(s3), c4, "bott")

    up = lambda t: layers.UpSampling2D(2, interpolation=UP_MODE)(t)
    d3 = block(layers.Concatenate()([up(b),  s3]), c3, "dec4")
    d2 = block(layers.Concatenate()([up(d3), s2]), c2, "dec3")
    d1 = block(layers.Concatenate()([up(d2), s1]), c1, "dec2")
    d0 = block(layers.Concatenate()([up(d1), s0]), c0, "dec1")

    if FAST_STEM:
        out = layers.Conv2D(1, 1, dtype="float32", name="head")(d0)
        out = layers.UpSampling2D(2, interpolation=UP_MODE, dtype="float32", name="logits")(out)
    else:
        out = layers.Conv2D(1, 1, dtype="float32", name="logits")(d0)
    return Model(inp, out, name="LiteUNet")

def count_params(m):
    return m.count_params() / 1e6

def count_gmacs(model):
    macs = 0
    for l in model.layers:
        if isinstance(l, layers.SeparableConv2D):
            h, w, cout = l.output.shape[1:4]; cin = l.input.shape[-1]; k = l.kernel_size[0]
            macs += h * w * cin * k * k + h * w * cin * cout
        elif isinstance(l, layers.Conv2D):
            h, w, cout = l.output.shape[1:4]; cin = l.input.shape[-1]; k = l.kernel_size[0]
            macs += h * w * cout * cin * k * k
    return macs / 1e9

# ------------------------------- DATA --------------------------------
def build_splits(img_dir, mask_dir, seed=SEED):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
    imgs  = sorted(f for f in os.listdir(img_dir) if f.lower().endswith(exts))
    masks = {os.path.splitext(f)[0]: f for f in os.listdir(mask_dir) if f.lower().endswith(exts)}
    pairs = [(os.path.join(img_dir, f), os.path.join(mask_dir, masks[os.path.splitext(f)[0]]))
             for f in imgs if os.path.splitext(f)[0] in masks]
    assert pairs, "No image/mask pairs found - check paths and filenames."
    rng = random.Random(seed); rng.shuffle(pairs)
    n = len(pairs); n_tr = int(0.8 * n); n_va = int(0.1 * n)
    return pairs[:n_tr], pairs[n_tr:n_tr + n_va], pairs[n_tr + n_va:]

def load_pair(ip, mp):
    img = cv2.imread(ip); msk = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    assert img is not None, f"Cannot read image: {ip}"
    assert msk is not None, f"Cannot read mask: {mp}"
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # INTER_AREA is the best interpolation for shrinking (keeps detail when compressing resolution)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    msk = cv2.resize(msk, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    return img, (msk > 127).astype(np.uint8)

# [COMPRESS] ---------------------------------------------------------
def compress_dataset(pairs, name):
    """Data compression step. Decodes + downsizes images ONCE, keeps them as uint8
    (4x smaller than float32), and stores them in a compressed .npz cache.
    Later runs load the cache in seconds instead of re-decoding every JPEG."""
    cache = f"{BASE}_cache_{name}_{IMG_SIZE}_{len(pairs)}.npz"
    if USE_CACHE and os.path.exists(cache):
        d = np.load(cache)
        print(f"[{name}] loaded compressed cache: {os.path.basename(cache)}")
        return d["imgs"], d["msks"]
    data = [load_pair(ip, mp) for ip, mp in tqdm(pairs, desc=f"compress {name}", leave=False)]
    imgs = np.stack([d[0] for d in data]).astype(np.uint8)
    msks = np.stack([d[1] for d in data]).astype(np.uint8)
    if USE_CACHE:
        np.savez_compressed(cache, imgs=imgs, msks=msks)
        print(f"[{name}] saved cache: {os.path.getsize(cache)/1024**2:.1f} MB "
              f"(raw uint8 {(imgs.nbytes + msks.nbytes)/1024**2:.1f} MB)")
    return imgs, msks


def normalize(img):
    return (img.astype(np.float32) / 255.0 - MEAN) / STD

# [COMPRESS] augmentation in pure TF ops (no cv2 / numpy_function -> no Python GIL bottleneck)
def tf_augment(img, msk):
    """img: uint8 (H,W,3), msk: uint8 (H,W). Returns normalized float32 image and float32 mask."""
    s = IMG_SIZE
    img = tf.cast(img, tf.float32); msk = tf.cast(msk, tf.float32)[..., None]
    # flips
    if_h = tf.random.uniform(()) < 0.5
    img, msk = tf.cond(if_h, lambda: (img[:, ::-1], msk[:, ::-1]), lambda: (img, msk))
    if_v = tf.random.uniform(()) < 0.5
    img, msk = tf.cond(if_v, lambda: (img[::-1], msk[::-1]), lambda: (img, msk))
    k = tf.random.uniform((), 0, 4, dtype=tf.int32)
    img, msk = tf.image.rot90(img, k), tf.image.rot90(msk, k)
    # random affine (rotation +-30deg, scale 0.85-1.2, shift +-10%), applied with prob 0.7
    ang = tf.random.uniform((), -30., 30.) * math.pi / 180.0
    sc  = tf.random.uniform((), 0.85, 1.2)
    tx  = tf.random.uniform((), -0.1, 0.1) * s
    ty  = tf.random.uniform((), -0.1, 0.1) * s
    c, sn, c0 = tf.cos(ang) / sc, tf.sin(ang) / sc, s / 2.0
    T = tf.stack([c, -sn, c0 - c * c0 + sn * c0 + tx,
                  sn,  c, c0 - sn * c0 - c * c0 + ty, 0., 0.])
    ident = tf.constant([1., 0., 0., 0., 1., 0., 0., 0.])
    T = tf.where(tf.random.uniform(()) < 0.7, T, ident)[None]
    def warp(t, mode):
        return tf.raw_ops.ImageProjectiveTransformV3(
            images=t[None], transforms=T, output_shape=[s, s], fill_value=0.0,
            interpolation=mode, fill_mode="REFLECT")[0]
    img, msk = warp(img, "BILINEAR"), warp(msk, "NEAREST")
    # brightness / contrast (prob 0.7)
    bc = tf.clip_by_value(img * tf.random.uniform((), 0.8, 1.2) + tf.random.uniform((), -20., 20.), 0., 255.)
    img = tf.where(tf.random.uniform(()) < 0.7, bc, img)
    # hue / saturation jitter (prob 0.3)
    hs = tf.image.random_saturation(tf.image.random_hue(img / 255.0, 0.04), 0.9, 1.1) * 255.0
    img = tf.where(tf.random.uniform(()) < 0.3, tf.clip_by_value(hs, 0., 255.), img)
    # normalize last, so stored data stays uint8
    img = (img / 255.0 - MEAN) / STD
    return img, tf.cast(msk > 0.5, tf.float32)

def make_train_ds(imgs, msks):
    n = len(imgs)
    ds = tf.data.Dataset.from_tensor_slices((imgs, msks))          # uint8 tensors (compressed in memory)
    return (ds.shuffle(n, reshuffle_each_iteration=True)
              .map(tf_augment, num_parallel_calls=AUTOTUNE)
              .batch(BATCH_SIZE, drop_remainder=True).prefetch(AUTOTUNE))

def prep_eval(imgs, msks):
    x = np.stack([normalize(i) for i in imgs])
    y = msks.astype(np.float32)[..., None]
    return x, y


# --------------------------- LOSS / METRICS --------------------------
def seg_loss(y_true, logits):
    y_true = tf.cast(y_true, tf.float32); logits = tf.cast(logits, tf.float32)
    bce = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=y_true, logits=logits))
    p = tf.reshape(tf.sigmoid(logits), (tf.shape(logits)[0], -1))
    g = tf.reshape(y_true, (tf.shape(logits)[0], -1))
    d = 1.0 - (2 * tf.reduce_sum(p * g, 1) + 1.0) / (tf.reduce_sum(p, 1) + tf.reduce_sum(g, 1) + 1.0)
    return bce + tf.reduce_mean(d)


def _bin(logits):
    return tf.cast(tf.sigmoid(tf.cast(logits, tf.float32)) > 0.5, tf.float32)


def dice(y_true, y_pred):
    p, g = _bin(y_pred), tf.cast(y_true, tf.float32)
    tp = tf.reduce_sum(p * g, (1, 2, 3))
    s = tf.reduce_sum(p, (1, 2, 3)) + tf.reduce_sum(g, (1, 2, 3))
    return tf.reduce_mean((2 * tp + 1e-7) / (s + 1e-7))


def iou(y_true, y_pred):
    p, g = _bin(y_pred), tf.cast(y_true, tf.float32)
    tp = tf.reduce_sum(p * g, (1, 2, 3))
    u = tf.reduce_sum(p, (1, 2, 3)) + tf.reduce_sum(g, (1, 2, 3)) - tp
    return tf.reduce_mean((tp + 1e-7) / (u + 1e-7))


def batch_metrics(logits, gt, thr=0.5, eps=1e-7):
    logits = np.clip(logits.astype(np.float64), -50, 50)
    p = (1 / (1 + np.exp(-logits)) > thr).astype(np.float64).reshape(len(gt), -1)
    g = gt.astype(np.float64).reshape(len(gt), -1)
    tp = (p * g).sum(1); fp = (p * (1 - g)).sum(1); fn = ((1 - p) * g).sum(1); tn = ((1 - p) * (1 - g)).sum(1)
    return {
        "dice":        (2 * tp + eps) / (2 * tp + fp + fn + eps),
        "iou":         (tp + eps) / (tp + fp + fn + eps),
        "precision":   (tp + eps) / (tp + fp + eps),
        "recall":      (tp + eps) / (tp + fn + eps),
        "specificity": (tn + eps) / (tn + fp + eps),
        "accuracy":    (tp + tn) / (tp + tn + fp + fn),
    }


def evaluate(model, x, y):
    logits = model.predict(x, batch_size=32, verbose=0)
    res = {k: float(v.mean()) for k, v in batch_metrics(logits, y).items()}
    res["loss"] = float(seg_loss(y, logits).numpy())
    return res


# ------------------------------ TRAINING -----------------------------
class OneCycle(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, max_lr, total_steps, pct=0.1, div=25.0, final=1e-6):
        super().__init__()
        self.max_lr, self.total, self.pct, self.div, self.final = max_lr, total_steps, pct, div, final

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warm = float(self.total) * self.pct
        start = self.max_lr / self.div
        lr_w = start + (self.max_lr - start) * step / warm
        prog = tf.clip_by_value((step - warm) / (float(self.total) - warm), 0.0, 1.0)
        lr_c = self.final + (self.max_lr - self.final) * 0.5 * (1.0 + tf.cos(math.pi * prog))
        return tf.where(step < warm, lr_w, lr_c)

    def get_config(self):
        return dict(max_lr=self.max_lr, total_steps=self.total, pct=self.pct, div=self.div, final=self.final)


class TrainMonitor(tf.keras.callbacks.Callback):
    """Progress %, ETA, all metrics every epoch, CSV log, best-model (.keras) saving, early stopping."""
    def __init__(self, xv, yv, steps_per_epoch, sched):
        super().__init__()
        self.xv, self.yv, self.spe, self.sched = xv, yv, steps_per_epoch, sched
        self.best, self.bad = -1.0, 0
        self.export = build_model()
        self.csv_path = BASE + "_log.csv"

    def on_train_begin(self, logs=None):
        self.t_start = time.time()
        with open(self.csv_path, "w", newline="") as f:
            csv.writer(f).writerow(["epoch", "lr", "train_loss", "train_dice", "train_iou", "val_loss"]
                                   + [f"val_{k}" for k in METRIC_KEYS] + ["best_val_dice"])

    def on_epoch_begin(self, epoch, logs=None):
        self.t0 = time.time()
        pct = 100.0 * epoch / EPOCHS
        self.pbar = tqdm(total=self.spe, desc=f"Ep {epoch+1:03d}/{EPOCHS} [{pct:5.1f}%]",
                         leave=False, ncols=100)

    def on_train_batch_end(self, batch, logs=None):
        self.pbar.update(1)
        if logs:
            self.pbar.set_postfix(loss=f"{logs.get('loss', 0.0):.4f}")

    def on_epoch_end(self, epoch, logs=None):
        self.pbar.close()
        logs = logs or {}
        v = evaluate(self.model, self.xv, self.yv)
        done = epoch + 1
        pct = 100.0 * done / EPOCHS
        elapsed = time.time() - self.t_start
        eta = elapsed / done * (EPOCHS - done)
        lr = float(self.sched(tf.constant(done * self.spe, dtype=tf.float32)))

        improved = v["dice"] > self.best
        if improved:
            self.best, self.bad = v["dice"], 0
            self.export.set_weights(self.model.get_weights())
            self.export.save(SAVE_PATH)
        else:
            self.bad += 1

        print(f"[Ep {done:03d}/{EPOCHS} | {pct:5.1f}%] {time.time()-self.t0:5.1f}s | "
              f"ETA {eta/60:6.1f} min | lr {lr:.2e}")
        print(f"   train: loss {logs.get('loss', float('nan')):.4f} | "
              f"dice {logs.get('dice', float('nan')):.4f} | iou {logs.get('iou', float('nan')):.4f}")
        print(f"   val  : loss {v['loss']:.4f} | dice {v['dice']:.4f} | iou {v['iou']:.4f} | "
              f"prec {v['precision']:.4f} | rec {v['recall']:.4f} | "
              f"spec {v['specificity']:.4f} | acc {v['accuracy']:.4f}")
        print(f"   best val dice {self.best:.4f}" + ("  <-- new best, model saved" if improved
                                                      else f"  (no gain for {self.bad}/{PATIENCE})"))

        with open(self.csv_path, "a", newline="") as f:
            csv.writer(f).writerow([done, lr, logs.get("loss"), logs.get("dice"), logs.get("iou"), v["loss"]]
                                   + [v[k] for k in METRIC_KEYS] + [self.best])

        if self.bad >= PATIENCE:
            print("Early stopping.")
            self.model.stop_training = True


# ------------------------------- FPS ---------------------------------
def _make_infer(model):
    @tf.function
    def infer(x):
        return tf.sigmoid(model(x, training=False))
    return infer


def measure_fps(model, batch=1, warmup=50, runs=300):
    infer = _make_infer(model)
    x = tf.constant(np.random.rand(batch, IMG_SIZE, IMG_SIZE, 3).astype(np.float32))
    for _ in range(warmup):
        infer(x).numpy()
    t0 = time.perf_counter()
    for _ in range(runs):
        infer(x).numpy()
    return batch * runs / (time.perf_counter() - t0)

def measure_fps_e2e(model, img_path, warmup=30, runs=200):
    """Full pipeline: resize + normalize + inference + threshold (on a real image)."""
    raw = cv2.imread(img_path)
    infer = _make_infer(model)

    def step():
        im = cv2.cvtColor(cv2.resize(raw, (IMG_SIZE, IMG_SIZE)), cv2.COLOR_BGR2RGB)
        x = tf.constant(normalize(im)[None])
        return infer(x).numpy()[0, ..., 0] > 0.5

    for _ in range(warmup):
        step()
    t0 = time.perf_counter()
    for _ in range(runs):
        step()
    return runs / (time.perf_counter() - t0)

# ------------------------------- MAIN --------------------------------
def train():
    seed_everything(SEED)
    os.makedirs(os.path.dirname(os.path.abspath(SAVE_PATH)), exist_ok=True)

    # let the GPU allocate memory gradually instead of grabbing all 8 GB at once
    for g in tf.config.list_physical_devices("GPU"):
        try: tf.config.experimental.set_memory_growth(g, True)
        except Exception: pass

    tr, va, te = build_splits(IMG_DIR, MASK_DIR)
    print(f"Split -> train {len(tr)} | val {len(va)} | test {len(te)}")
    with open(BASE + "_splits.json", "w") as f:
        json.dump({"train": tr, "val": va, "test": te}, f, indent=1)

    # [COMPRESS] compressed, cached uint8 arrays for every split
    tr_i, tr_m = compress_dataset(tr, "train")
    va_i, va_m = compress_dataset(va, "val")
    te_i, te_m = compress_dataset(te, "test")
    train_ds = make_train_ds(tr_i, tr_m)
    xv, yv = prep_eval(va_i, va_m)
    xt, yt = prep_eval(te_i, te_m)

    gpu = len(tf.config.list_physical_devices("GPU")) > 0
    print("Device:", "GPU" if gpu else "CPU (no GPU detected)")
    tf.keras.mixed_precision.set_global_policy("mixed_float16" if (gpu and USE_MIXED_PRECISION) else "float32")

    model = build_model()
    print(f"Params: {count_params(model):.4f} M | GMACs: {count_gmacs(model):.3f}")

    spe = len(tr_i) // BATCH_SIZE
    sched = OneCycle(MAX_LR, spe * EPOCHS)
    opt = tf.keras.optimizers.AdamW(sched, weight_decay=WEIGHT_DECAY, global_clipnorm=5.0)
    model.compile(optimizer=opt, loss=seg_loss, metrics=[dice, iou], jit_compile=False)
    model.fit(train_ds, epochs=EPOCHS, verbose=0, callbacks=[TrainMonitor(xv, yv, spe, sched)])

    # ---- reload the best saved .keras model and test it
    model = tf.keras.models.load_model(SAVE_PATH, compile=False)
    res = evaluate(model, xt, yt)
    params, gmacs = count_params(model), count_gmacs(model)
    size_mb = os.path.getsize(SAVE_PATH) / 1024 ** 2
    fps = measure_fps(model)
    fps_e2e = measure_fps_e2e(model, te[0][0])

    print("\n================ TEST RESULTS ================")
    rows = [("GMACs", gmacs, f"< {TARGETS['gmacs']}", gmacs < TARGETS["gmacs"]),
            ("Params (M)", params, f"<= {TARGETS['params_m']}", params <= TARGETS["params_m"]),
            ("Size (MB)", size_mb, f"< {TARGETS['size_mb']}", size_mb < TARGETS["size_mb"]),
            ("FPS (model)", fps, f">= {TARGETS['fps']:.0f}", fps >= TARGETS["fps"])]
    for k in METRIC_KEYS:
        rows.append((k, res[k], f"> {TARGETS['metric']}", res[k] > TARGETS["metric"]))
    for name, val, tgt, ok in rows:
        print(f"{name:<12} {val:10.4f}   target {tgt:<8}  {'PASS' if ok else 'FAIL'}")
    print(f"{'FPS (e2e)':<12} {fps_e2e:10.4f}   (resize + normalize + model + threshold)")
    out = {r[0]: float(r[1]) for r in rows}
    out["FPS (e2e)"] = float(fps_e2e)
    with open(BASE + "_test_metrics.json", "w") as f:
        json.dump(out, f, indent=1)

    explain(model, xt, yt, os.path.join(os.path.dirname(os.path.abspath(SAVE_PATH)), "xai"))

# --------------------------- EXPLAINABILITY --------------------------
def _norm(c):
    c = c - c.min()
    return c / (c.max() + 1e-8)
def _resize(t):
    t = t if t.shape.rank == 4 else t[None]
    return tf.image.resize(t, (IMG_SIZE, IMG_SIZE), method="bilinear")[0]
def grad_cam(model, x, layer_name):
    gm = Model(model.inputs, [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        act, logits = gm(x, training=False)
        mask = tf.cast(tf.sigmoid(logits) > 0.5, tf.float32)
        if tf.reduce_sum(mask) == 0: mask = tf.ones_like(mask)
        target = tf.reduce_sum(logits * mask)
    grads = tape.gradient(target, act)
    w = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(w * act, -1, keepdims=True))
    return _norm(_resize(cam).numpy()[..., 0])
def score_cam(model, x, layer_name, chunk=32):
    gm = Model(model.inputs, [model.get_layer(layer_name).output, model.output])
    act, logits = gm(x, training=False)
    mask = tf.cast(tf.sigmoid(logits) > 0.5, tf.float32)
    if tf.reduce_sum(mask) == 0: mask = tf.ones_like(mask)
    up = _resize(act).numpy()
    up = (up - up.min((0, 1), keepdims=True)) / (up.max((0, 1), keepdims=True) - up.min((0, 1), keepdims=True) + 1e-8)
    up = tf.constant(np.transpose(up, (2, 0, 1))[..., None].astype(np.float32))
    scores = []
    for i in range(0, up.shape[0], chunk):
        p = tf.sigmoid(model(x * up[i:i + chunk], training=False))
        scores.append(tf.reduce_sum(p * mask, (1, 2, 3)) / tf.reduce_sum(mask))
    w = tf.nn.softmax(tf.concat(scores, 0))
    cam = tf.nn.relu(tf.reduce_sum(w * act[0], -1, keepdims=True))
    return _norm(_resize(cam).numpy()[..., 0])
def denorm(x):
    return np.clip(x * STD + MEAN, 0, 1)
def overlay(img, cam):
    heat = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)[..., ::-1] / 255.0
    return np.clip(0.55 * img + 0.45 * heat, 0, 1)

def save_feature_maps(model, x, path, n=16, names=("stem_out", "enc2_out", "enc3_out", "dec3_out")):
    fm = Model(model.inputs, [model.get_layer(k).output for k in names])
    feats = [f[0].numpy() for f in fm(x, training=False)]
    fig, axes = plt.subplots(len(names), n, figsize=(n * 1.1, len(names) * 1.2))
    for r, (k, f) in enumerate(zip(names, feats)):
        for c in range(n):
            ax = axes[r, c]; ax.imshow(f[..., c], cmap="viridis"); ax.axis("off")
            if c == 0: ax.set_title(k, fontsize=7, loc="left")
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()


def explain(model, xs, ys, out_dir, layer_name="dec3_out"):
    os.makedirs(out_dir, exist_ok=True)
    idx = np.linspace(0, len(xs) - 1, min(N_XAI_IMAGES, len(xs))).astype(int)
    for j, i in enumerate(idx):
        x = xs[i][None]
        pred = (tf.sigmoid(model(x, training=False)).numpy()[0, ..., 0] > 0.5)
        img = denorm(xs[i])
        g, s = grad_cam(model, x, layer_name), score_cam(model, x, layer_name)
        fig, ax = plt.subplots(1, 5, figsize=(13, 2.8))
        for a, im, t in zip(ax, [img, ys[i][..., 0], pred, overlay(img, g), overlay(img, s)],
                            ["Image", "Ground truth", "Prediction", "Grad-CAM", "Score-CAM"]):
            a.imshow(im, cmap="gray" if im.ndim == 2 else None); a.set_title(t, fontsize=9); a.axis("off")
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, f"cam_{j:02d}.png"), dpi=150); plt.close()
        if j == 0:
            save_feature_maps(model, x, os.path.join(out_dir, "feature_maps.png"))
    print(f"XAI figures saved to: {out_dir}")


In [30]:
if __name__ == "__main__":
    train()

Split -> train 800 | val 100 | test 100


[train] saved cache: 45.3 MB (raw uint8 78.1 MB)


[val] saved cache: 5.6 MB (raw uint8 9.8 MB)


[test] saved cache: 5.7 MB (raw uint8 9.8 MB)
Device: GPU
Params: 0.2457 M | GMACs: 0.124


[Ep 001/100 |   1.0%]  36.5s | ETA   59.6 min | lr 4.08e-04
   train: loss 1.2862 | dice 0.3875 | iou 0.2645
   val  : loss 1.4036 | dice 0.0000 | iou 0.0000 | prec 1.0000 | rec 0.0000 | spec 1.0000 | acc 0.8193
   best val dice 0.0000  <-- new best, model saved


...

[Ep 098/100 |  98.0%]  21.5s | ETA    0.8 min | lr 4.65e-06
   train: loss 0.2841 | dice 0.8615 | iou 0.7839
   val  : loss 0.3441 | dice 0.8461 | iou 0.7728 | prec 0.9008 | rec 0.8468 | spec 0.9823 | acc 0.9536
   best val dice 0.8484  (no gain for 8/20)


[Ep 099/100 |  99.0%]  21.9s | ETA    0.4 min | lr 1.91e-06
   train: loss 0.2740 | dice 0.8625 | iou 0.7861
   val  : loss 0.3439 | dice 0.8460 | iou 0.7727 | prec 0.9013 | rec 0.8463 | spec 0.9825 | acc 0.9535
   best val dice 0.8484  (no gain for 9/20)


[Ep 100/100 | 100.0%]  21.8s | ETA    0.0 min | lr 1.00e-06
   train: loss 0.2637 | dice 0.8698 | iou 0.7956
   val  : loss 0.3432 | dice 0.8462 | iou 0.7731 | prec 0.9002 | rec 0.8472 | spec 0.9822 | acc 0.9536
   best val dice 0.8484  (no gain for 10/20)

================ TEST RESULTS ================
GMACs            0.1245   target < 1.0     PASS
Params (M)       0.2457   target <= 0.5    PASS
Size (MB)        1.1861   target < 2.0     PASS
FPS (model)     64.3436   target >= 50     PASS
dice             0.8507   target > 0.95    FAIL
iou              0.7808   target > 0.95    FAIL
precision        0.8701   target > 0.95    FAIL
recall           0.8739   target > 0.95    FAIL
specificity      0.9703   target > 0.95    PASS
accuracy         0.9554   target > 0.95    PASS
FPS (e2e)       64.4901   (resize + normalize + model + threshold)


/mnt/c/development/Thesis/PotatoDiseaseClassification-CNN/linux-venv/lib/python3.12/site-packages/keras/src/models/functional.py:258: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input_layer_5']
Received: inputs=Tensor(shape=(1, 160, 160, 3))
  warnings.warn(msg)


XAI figures saved to: /mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/SC/tr-elm/xai


In [20]:
import re
import matplotlib.pyplot as plt

# Your training information string

GRAPH_TITLE = "Training Time per Epoch"

# Extract epochs and times using Regular Expressions
epochs = []
times = []
pattern = r"\[Ep\s+(\d+)/\d+\s+\|[^\]]+\]\s+([\d\.]+)s"
matches = re.findall(pattern, training_info)

for ep, t in matches:
    epochs.append(int(ep))
    times.append(float(t))

# Create the connecting point graph (make the figure nice and wide)
plt.figure(figsize=(12, 5))
plt.plot(epochs, times, marker='o', linestyle='-', color='tab:blue', linewidth=2, markersize=5)

# Set labels and title using the variable
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Time (seconds)", fontsize=12)
plt.title(GRAPH_TITLE, fontsize=14, fontweight='bold')

plt.grid(True, linestyle='--', alpha=0.6)

# --- FIX FOR OVERLAPPING TICKS ---
# Automatically calculate a step size so labels never crowd each other (aims for ~15-20 labels max)
tick_step = max(1, len(epochs) // 15) 
plt.xticks(epochs[::tick_step], rotation=45, ha='right')

plt.tight_layout()

# Save the figure
plt.savefig("training_time_plot.png", dpi=300)
print("Graph saved successfully as 'training_time_plot.png'!")


Graph saved successfully as 'training_time_plot.png'!


In [31]:
training_compressed_info = """
Split -> train 800 | val 100 | test 100
                                                                                                                                                               
[train] saved cache: 45.3 MB (raw uint8 78.1 MB)
                                                                                                                                                               
[val] saved cache: 5.6 MB (raw uint8 9.8 MB)
                                                                                                                                                               
[test] saved cache: 5.7 MB (raw uint8 9.8 MB)
Device: GPU
Params: 0.2457 M | GMACs: 0.124
                                                                                                    
WARNING:tensorflow:5 out of the last 245 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x7eb9b8eba840> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
[Ep 001/100 |   1.0%]  36.5s | ETA   59.6 min | lr 4.08e-04
   train: loss 1.2862 | dice 0.3875 | iou 0.2645
   val  : loss 1.4036 | dice 0.0000 | iou 0.0000 | prec 1.0000 | rec 0.0000 | spec 1.0000 | acc 0.8193
   best val dice 0.0000  <-- new best, model saved
                                                                                                    
[Ep 002/100 |   2.0%]  20.5s | ETA   46.5 min | lr 6.96e-04
   train: loss 1.0722 | dice 0.4533 | iou 0.3233
   val  : loss 1.3115 | dice 0.0000 | iou 0.0000 | prec 1.0000 | rec 0.0000 | spec 1.0000 | acc 0.8193
   best val dice 0.0000  (no gain for 1/20)
                                                                                                    
[Ep 003/100 |   3.0%]  19.8s | ETA   41.3 min | lr 9.84e-04
   train: loss 0.9736 | dice 0.4710 | iou 0.3468
   val  : loss 1.2351 | dice 0.0000 | iou 0.0000 | prec 1.0000 | rec 0.0000 | spec 1.0000 | acc 0.8193
   best val dice 0.0000  (no gain for 2/20)
                                                                                                    
[Ep 004/100 |   4.0%]  19.9s | ETA   38.6 min | lr 1.27e-03
   train: loss 0.9119 | dice 0.4987 | iou 0.3722
   val  : loss 1.3623 | dice 0.0000 | iou 0.0000 | prec 1.0000 | rec 0.0000 | spec 1.0000 | acc 0.8193
   best val dice 0.0000  (no gain for 3/20)
                                                                                                    
[Ep 005/100 |   5.0%]  23.1s | ETA   37.8 min | lr 1.56e-03
   train: loss 0.8532 | dice 0.5409 | iou 0.4145
   val  : loss 1.0405 | dice 0.3844 | iou 0.2903 | prec 0.8435 | rec 0.3149 | spec 0.9950 | acc 0.8553
   best val dice 0.3844  <-- new best, model saved
                                                                                                    
[Ep 006/100 |   6.0%]  21.2s | ETA   36.7 min | lr 1.85e-03
   train: loss 0.7917 | dice 0.5830 | iou 0.4580
   val  : loss 0.8083 | dice 0.6168 | iou 0.4906 | prec 0.6490 | rec 0.6984 | spec 0.9324 | acc 0.8711
   best val dice 0.6168  <-- new best, model saved
                                                                                                    
[Ep 007/100 |   7.0%]  22.2s | ETA   36.1 min | lr 2.14e-03
   train: loss 0.7727 | dice 0.5960 | iou 0.4773
   val  : loss 0.9462 | dice 0.5353 | iou 0.4250 | prec 0.7879 | rec 0.4932 | spec 0.9805 | acc 0.8747
   best val dice 0.6168  (no gain for 1/20)
                                                                                                    
[Ep 008/100 |   8.0%]  21.1s | ETA   35.3 min | lr 2.42e-03
   train: loss 0.7224 | dice 0.6280 | iou 0.5070
   val  : loss 1.0457 | dice 0.5822 | iou 0.4720 | prec 0.8214 | rec 0.5251 | spec 0.9836 | acc 0.8799
   best val dice 0.6168  (no gain for 2/20)
                                                                                                    
[Ep 009/100 |   9.0%]  20.5s | ETA   34.5 min | lr 2.71e-03
   train: loss 0.6625 | dice 0.6731 | iou 0.5591
   val  : loss 1.0165 | dice 0.5860 | iou 0.4540 | prec 0.5711 | rec 0.7293 | spec 0.8769 | acc 0.8369
   best val dice 0.6168  (no gain for 3/20)
                                                                                                    
[Ep 010/100 |  10.0%]  21.6s | ETA   33.9 min | lr 3.00e-03
   train: loss 0.6478 | dice 0.6739 | iou 0.5622
   val  : loss 1.2334 | dice 0.6191 | iou 0.4895 | prec 0.5471 | rec 0.8623 | spec 0.8121 | acc 0.8150
   best val dice 0.6191  <-- new best, model saved
                                                                                                    
[Ep 011/100 |  11.0%]  22.4s | ETA   33.5 min | lr 3.00e-03
   train: loss 0.6371 | dice 0.6816 | iou 0.5707
   val  : loss 1.3714 | dice 0.5480 | iou 0.4289 | prec 0.7696 | rec 0.4910 | spec 0.9735 | acc 0.8700
   best val dice 0.6191  (no gain for 1/20)
                                                                                                    
[Ep 012/100 |  12.0%]  21.7s | ETA   33.0 min | lr 3.00e-03
   train: loss 0.5783 | dice 0.7122 | iou 0.6030
   val  : loss 0.8557 | dice 0.6401 | iou 0.5319 | prec 0.8861 | rec 0.5710 | spec 0.9881 | acc 0.8927
   best val dice 0.6401  <-- new best, model saved
                                                                                                    
[Ep 013/100 |  13.0%]  20.7s | ETA   32.4 min | lr 2.99e-03
   train: loss 0.5867 | dice 0.7128 | iou 0.6046
   val  : loss 0.6541 | dice 0.7310 | iou 0.6202 | prec 0.7542 | rec 0.8040 | spec 0.9337 | acc 0.8981
   best val dice 0.7310  <-- new best, model saved
                                                                                                    
[Ep 014/100 |  14.0%]  21.7s | ETA   32.0 min | lr 2.99e-03
   train: loss 0.5790 | dice 0.7202 | iou 0.6142
   val  : loss 1.0550 | dice 0.6351 | iou 0.5063 | prec 0.5727 | rec 0.8612 | spec 0.8245 | acc 0.8218
   best val dice 0.7310  (no gain for 1/20)
                                                                                                    
[Ep 015/100 |  15.0%]  20.8s | ETA   31.5 min | lr 2.98e-03
   train: loss 0.5568 | dice 0.7286 | iou 0.6239
   val  : loss 0.6307 | dice 0.6995 | iou 0.5855 | prec 0.6896 | rec 0.8277 | spec 0.9027 | acc 0.8810
   best val dice 0.7310  (no gain for 2/20)
                                                                                                    
[Ep 016/100 |  16.0%]  22.9s | ETA   31.2 min | lr 2.97e-03
   train: loss 0.5582 | dice 0.7262 | iou 0.6230
   val  : loss 0.6980 | dice 0.6863 | iou 0.5869 | prec 0.7941 | rec 0.6725 | spec 0.9715 | acc 0.9031
   best val dice 0.7310  (no gain for 3/20)
                                                                                                    
[Ep 017/100 |  17.0%]  22.7s | ETA   30.8 min | lr 2.96e-03
   train: loss 0.5580 | dice 0.7284 | iou 0.6248
   val  : loss 0.5412 | dice 0.7684 | iou 0.6700 | prec 0.8681 | rec 0.7457 | spec 0.9812 | acc 0.9219
   best val dice 0.7684  <-- new best, model saved
                                                                                                    
[Ep 018/100 |  18.0%]  28.6s | ETA   31.0 min | lr 2.94e-03
   train: loss 0.5190 | dice 0.7449 | iou 0.6403
   val  : loss 0.6222 | dice 0.7428 | iou 0.6502 | prec 0.8814 | rec 0.7197 | spec 0.9822 | acc 0.9214
   best val dice 0.7684  (no gain for 1/20)
                                                                                                    
[Ep 019/100 |  19.0%]  25.3s | ETA   30.8 min | lr 2.93e-03
   train: loss 0.5292 | dice 0.7413 | iou 0.6378
   val  : loss 0.6165 | dice 0.7256 | iou 0.6216 | prec 0.7669 | rec 0.7634 | spec 0.9491 | acc 0.9022
   best val dice 0.7684  (no gain for 2/20)
                                                                                                    
[Ep 020/100 |  20.0%]  22.8s | ETA   30.4 min | lr 2.91e-03
   train: loss 0.5274 | dice 0.7487 | iou 0.6479
   val  : loss 0.6312 | dice 0.7368 | iou 0.6314 | prec 0.8318 | rec 0.7345 | spec 0.9685 | acc 0.9086
   best val dice 0.7684  (no gain for 3/20)
                                                                                                    
[Ep 021/100 |  21.0%]  22.9s | ETA   30.0 min | lr 2.89e-03
   train: loss 0.5085 | dice 0.7505 | iou 0.6516
   val  : loss 0.6092 | dice 0.7385 | iou 0.6229 | prec 0.6866 | rec 0.8913 | spec 0.9016 | acc 0.8933
   best val dice 0.7684  (no gain for 4/20)
                                                                                                    
[Ep 022/100 |  22.0%]  23.3s | ETA   29.7 min | lr 2.87e-03
   train: loss 0.4992 | dice 0.7589 | iou 0.6592
   val  : loss 0.4787 | dice 0.7871 | iou 0.6884 | prec 0.8310 | rec 0.8057 | spec 0.9600 | acc 0.9258
   best val dice 0.7871  <-- new best, model saved
                                                                                                    
[Ep 023/100 |  23.0%]  22.2s | ETA   29.3 min | lr 2.85e-03
   train: loss 0.4738 | dice 0.7746 | iou 0.6756
   val  : loss 0.5663 | dice 0.7430 | iou 0.6458 | prec 0.8344 | rec 0.7375 | spec 0.9681 | acc 0.9144
   best val dice 0.7871  (no gain for 1/20)
                                                                                                    
[Ep 024/100 |  24.0%]  21.8s | ETA   28.8 min | lr 2.82e-03
   train: loss 0.4962 | dice 0.7624 | iou 0.6634
   val  : loss 0.5180 | dice 0.7458 | iou 0.6403 | prec 0.7674 | rec 0.8111 | spec 0.9398 | acc 0.9097
   best val dice 0.7871  (no gain for 2/20)
                                                                                                    
[Ep 025/100 |  25.0%]  20.5s | ETA   28.3 min | lr 2.80e-03
   train: loss 0.4750 | dice 0.7716 | iou 0.6744
   val  : loss 0.5167 | dice 0.7728 | iou 0.6679 | prec 0.7897 | rec 0.8345 | spec 0.9413 | acc 0.9131
   best val dice 0.7871  (no gain for 3/20)
                                                                                                    
[Ep 026/100 |  26.0%]  21.8s | ETA   27.9 min | lr 2.77e-03
   train: loss 0.4999 | dice 0.7584 | iou 0.6593
   val  : loss 0.4800 | dice 0.7756 | iou 0.6783 | prec 0.8439 | rec 0.7831 | spec 0.9679 | acc 0.9265
   best val dice 0.7871  (no gain for 4/20)
                                                                                                    
[Ep 027/100 |  27.0%]  21.1s | ETA   27.5 min | lr 2.74e-03
   train: loss 0.4602 | dice 0.7738 | iou 0.6768
   val  : loss 0.5031 | dice 0.7641 | iou 0.6631 | prec 0.7713 | rec 0.8436 | spec 0.9410 | acc 0.9192
   best val dice 0.7871  (no gain for 5/20)
                                                                                                    
[Ep 028/100 |  28.0%]  24.2s | ETA   27.2 min | lr 2.71e-03
   train: loss 0.4628 | dice 0.7769 | iou 0.6808
   val  : loss 0.5523 | dice 0.7449 | iou 0.6557 | prec 0.9200 | rec 0.6875 | spec 0.9922 | acc 0.9233
   best val dice 0.7871  (no gain for 6/20)
                                                                                                    
[Ep 029/100 |  29.0%]  30.1s | ETA   27.1 min | lr 2.68e-03
   train: loss 0.4547 | dice 0.7797 | iou 0.6821
   val  : loss 0.6055 | dice 0.7264 | iou 0.6229 | prec 0.7524 | rec 0.8046 | spec 0.9383 | acc 0.9059
   best val dice 0.7871  (no gain for 7/20)
                                                                                                    
[Ep 030/100 |  30.0%]  24.9s | ETA   26.8 min | lr 2.65e-03
   train: loss 0.4641 | dice 0.7760 | iou 0.6819
   val  : loss 0.5569 | dice 0.7603 | iou 0.6528 | prec 0.7517 | rec 0.8546 | spec 0.9293 | acc 0.9107
   best val dice 0.7871  (no gain for 8/20)
                                                                                                    
[Ep 031/100 |  31.0%]  22.6s | ETA   26.4 min | lr 2.61e-03
   train: loss 0.4439 | dice 0.7874 | iou 0.6924
   val  : loss 0.6526 | dice 0.7168 | iou 0.6164 | prec 0.8478 | rec 0.6980 | spec 0.9715 | acc 0.9097
   best val dice 0.7871  (no gain for 9/20)
                                                                                                    
[Ep 032/100 |  32.0%]  21.2s | ETA   25.9 min | lr 2.58e-03
   train: loss 0.4542 | dice 0.7861 | iou 0.6909
   val  : loss 0.4416 | dice 0.7977 | iou 0.7020 | prec 0.8462 | rec 0.8124 | spec 0.9661 | acc 0.9296
   best val dice 0.7977  <-- new best, model saved
                                                                                                    
[Ep 033/100 |  33.0%]  23.3s | ETA   25.6 min | lr 2.54e-03
   train: loss 0.4542 | dice 0.7841 | iou 0.6884
   val  : loss 1.4040 | dice 0.5470 | iou 0.4174 | prec 0.4256 | rec 0.9770 | spec 0.6525 | acc 0.7067
   best val dice 0.7977  (no gain for 1/20)
                                                                                                    
[Ep 034/100 |  34.0%]  25.1s | ETA   25.3 min | lr 2.50e-03
   train: loss 0.4303 | dice 0.7942 | iou 0.7006
   val  : loss 0.7300 | dice 0.6893 | iou 0.5696 | prec 0.6135 | rec 0.9055 | spec 0.8500 | acc 0.8629
   best val dice 0.7977  (no gain for 2/20)
                                                                                                    
[Ep 035/100 |  35.0%]  21.9s | ETA   24.9 min | lr 2.46e-03
   train: loss 0.4401 | dice 0.7914 | iou 0.6991
   val  : loss 0.4709 | dice 0.7714 | iou 0.6807 | prec 0.8362 | rec 0.7797 | spec 0.9739 | acc 0.9315
   best val dice 0.7977  (no gain for 3/20)
                                                                                                    
[Ep 036/100 |  36.0%]  23.6s | ETA   24.5 min | lr 2.42e-03
   train: loss 0.4340 | dice 0.7922 | iou 0.6997
   val  : loss 0.4312 | dice 0.8119 | iou 0.7205 | prec 0.8653 | rec 0.8152 | spec 0.9737 | acc 0.9359
   best val dice 0.8119  <-- new best, model saved
                                                                                                    
[Ep 037/100 |  37.0%]  20.9s | ETA   24.1 min | lr 2.38e-03
   train: loss 0.4110 | dice 0.8029 | iou 0.7119
   val  : loss 0.4513 | dice 0.8064 | iou 0.7158 | prec 0.8818 | rec 0.7980 | spec 0.9775 | acc 0.9362
   best val dice 0.8119  (no gain for 1/20)
                                                                                                    
[Ep 038/100 |  38.0%]  23.4s | ETA   23.7 min | lr 2.34e-03
   train: loss 0.4158 | dice 0.8008 | iou 0.7066
   val  : loss 0.4842 | dice 0.7755 | iou 0.6909 | prec 0.8489 | rec 0.7837 | spec 0.9712 | acc 0.9272
   best val dice 0.8119  (no gain for 2/20)
                                                                                                    
[Ep 039/100 |  39.0%]  24.0s | ETA   23.3 min | lr 2.30e-03
   train: loss 0.4128 | dice 0.7977 | iou 0.7030
   val  : loss 0.4673 | dice 0.7942 | iou 0.7083 | prec 0.8345 | rec 0.8203 | spec 0.9591 | acc 0.9283
   best val dice 0.8119  (no gain for 3/20)
                                                                                                    
[Ep 040/100 |  40.0%]  20.8s | ETA   22.9 min | lr 2.25e-03
   train: loss 0.4129 | dice 0.7976 | iou 0.7059
   val  : loss 0.4891 | dice 0.7742 | iou 0.6856 | prec 0.8660 | rec 0.7731 | spec 0.9715 | acc 0.9285
   best val dice 0.8119  (no gain for 4/20)
                                                                                                    
[Ep 041/100 |  41.0%]  22.5s | ETA   22.5 min | lr 2.20e-03
   train: loss 0.4224 | dice 0.7986 | iou 0.7063
   val  : loss 0.4885 | dice 0.7704 | iou 0.6859 | prec 0.8754 | rec 0.7617 | spec 0.9765 | acc 0.9303
   best val dice 0.8119  (no gain for 5/20)
                                                                                                    
[Ep 042/100 |  42.0%]  21.3s | ETA   22.1 min | lr 2.16e-03
   train: loss 0.4087 | dice 0.8031 | iou 0.7133
   val  : loss 0.4696 | dice 0.7802 | iou 0.6844 | prec 0.7737 | rec 0.8658 | spec 0.9391 | acc 0.9225
   best val dice 0.8119  (no gain for 6/20)
                                                                                                    
[Ep 043/100 |  43.0%]  22.1s | ETA   21.7 min | lr 2.11e-03
   train: loss 0.4036 | dice 0.8053 | iou 0.7133
   val  : loss 0.4956 | dice 0.7714 | iou 0.6664 | prec 0.7466 | rec 0.8849 | spec 0.9281 | acc 0.9146
   best val dice 0.8119  (no gain for 7/20)
                                                                                                    
[Ep 044/100 |  44.0%]  23.3s | ETA   21.3 min | lr 2.06e-03
   train: loss 0.3934 | dice 0.8113 | iou 0.7210
   val  : loss 0.4882 | dice 0.7903 | iou 0.7076 | prec 0.8941 | rec 0.7742 | spec 0.9849 | acc 0.9339
   best val dice 0.8119  (no gain for 8/20)
                                                                                                    
[Ep 045/100 |  45.0%]  21.7s | ETA   20.9 min | lr 2.01e-03
   train: loss 0.4148 | dice 0.7992 | iou 0.7070
   val  : loss 0.4255 | dice 0.8046 | iou 0.7131 | prec 0.8355 | rec 0.8506 | spec 0.9638 | acc 0.9349
   best val dice 0.8119  (no gain for 9/20)
                                                                                                    
[Ep 046/100 |  46.0%]  20.7s | ETA   20.5 min | lr 1.96e-03
   train: loss 0.3875 | dice 0.8169 | iou 0.7278
   val  : loss 0.4571 | dice 0.8054 | iou 0.7231 | prec 0.9174 | rec 0.7750 | spec 0.9879 | acc 0.9381
   best val dice 0.8119  (no gain for 10/20)
                                                                                                    
[Ep 047/100 |  47.0%]  23.0s | ETA   20.1 min | lr 1.91e-03
   train: loss 0.3848 | dice 0.8137 | iou 0.7219
   val  : loss 0.4258 | dice 0.8166 | iou 0.7294 | prec 0.8657 | rec 0.8297 | spec 0.9732 | acc 0.9358
   best val dice 0.8166  <-- new best, model saved
                                                                                                    
[Ep 048/100 |  48.0%]  22.6s | ETA   19.7 min | lr 1.86e-03
   train: loss 0.3727 | dice 0.8194 | iou 0.7311
   val  : loss 0.4278 | dice 0.8007 | iou 0.7121 | prec 0.8362 | rec 0.8382 | spec 0.9554 | acc 0.9301
   best val dice 0.8166  (no gain for 1/20)
                                                                                                    
[Ep 049/100 |  49.0%]  23.0s | ETA   19.4 min | lr 1.81e-03
   train: loss 0.3778 | dice 0.8162 | iou 0.7290
   val  : loss 0.4176 | dice 0.8293 | iou 0.7466 | prec 0.8699 | rec 0.8433 | spec 0.9733 | acc 0.9418
   best val dice 0.8293  <-- new best, model saved
                                                                                                    
[Ep 050/100 |  50.0%]  23.3s | ETA   19.0 min | lr 1.76e-03
   train: loss 0.3796 | dice 0.8171 | iou 0.7291
   val  : loss 0.4254 | dice 0.8164 | iou 0.7347 | prec 0.8597 | rec 0.8375 | spec 0.9679 | acc 0.9347
   best val dice 0.8293  (no gain for 1/20)
                                                                                                    
[Ep 051/100 |  51.0%]  24.4s | ETA   18.6 min | lr 1.71e-03
   train: loss 0.3620 | dice 0.8249 | iou 0.7353
   val  : loss 0.4130 | dice 0.8106 | iou 0.7273 | prec 0.8672 | rec 0.8249 | spec 0.9724 | acc 0.9370
   best val dice 0.8293  (no gain for 2/20)
                                                                                                    
[Ep 052/100 |  52.0%]  21.1s | ETA   18.2 min | lr 1.66e-03
   train: loss 0.3692 | dice 0.8204 | iou 0.7345
   val  : loss 0.4581 | dice 0.8003 | iou 0.7190 | prec 0.9177 | rec 0.7676 | spec 0.9869 | acc 0.9421
   best val dice 0.8293  (no gain for 3/20)
                                                                                                    
[Ep 053/100 |  53.0%]  19.9s | ETA   17.8 min | lr 1.61e-03
   train: loss 0.3616 | dice 0.8269 | iou 0.7408
   val  : loss 0.3990 | dice 0.8228 | iou 0.7369 | prec 0.8678 | rec 0.8429 | spec 0.9652 | acc 0.9391
   best val dice 0.8293  (no gain for 4/20)
                                                                                                    
[Ep 054/100 |  54.0%]  21.1s | ETA   17.4 min | lr 1.55e-03
   train: loss 0.3609 | dice 0.8254 | iou 0.7384
   val  : loss 0.3871 | dice 0.8172 | iou 0.7362 | prec 0.8994 | rec 0.8051 | spec 0.9833 | acc 0.9423
   best val dice 0.8293  (no gain for 5/20)
                                                                                                    
[Ep 055/100 |  55.0%]  22.3s | ETA   17.0 min | lr 1.50e-03
   train: loss 0.3653 | dice 0.8229 | iou 0.7343
   val  : loss 0.3981 | dice 0.8162 | iou 0.7327 | prec 0.8699 | rec 0.8369 | spec 0.9759 | acc 0.9405
   best val dice 0.8293  (no gain for 6/20)
                                                                                                    
[Ep 056/100 |  56.0%]  24.7s | ETA   16.7 min | lr 1.45e-03
   train: loss 0.3528 | dice 0.8308 | iou 0.7453
   val  : loss 0.3932 | dice 0.8230 | iou 0.7386 | prec 0.8910 | rec 0.8244 | spec 0.9804 | acc 0.9404
   best val dice 0.8293  (no gain for 7/20)
                                                                                                    
[Ep 057/100 |  57.0%]  26.9s | ETA   16.3 min | lr 1.40e-03
   train: loss 0.3549 | dice 0.8323 | iou 0.7476
   val  : loss 0.3893 | dice 0.8185 | iou 0.7366 | prec 0.8982 | rec 0.8005 | spec 0.9856 | acc 0.9417
   best val dice 0.8293  (no gain for 8/20)
                                                                                                    
[Ep 058/100 |  58.0%]  24.4s | ETA   16.0 min | lr 1.34e-03
   train: loss 0.3475 | dice 0.8318 | iou 0.7477
   val  : loss 0.4247 | dice 0.8032 | iou 0.7184 | prec 0.8933 | rec 0.7990 | spec 0.9846 | acc 0.9380
   best val dice 0.8293  (no gain for 9/20)
                                                                                                    
[Ep 059/100 |  59.0%]  23.9s | ETA   15.6 min | lr 1.29e-03
   train: loss 0.3515 | dice 0.8266 | iou 0.7423
   val  : loss 0.4417 | dice 0.8020 | iou 0.7206 | prec 0.8927 | rec 0.7859 | spec 0.9852 | acc 0.9405
   best val dice 0.8293  (no gain for 10/20)
                                                                                                    
[Ep 060/100 |  60.0%]  21.8s | ETA   15.2 min | lr 1.24e-03
   train: loss 0.3458 | dice 0.8354 | iou 0.7506
   val  : loss 0.4423 | dice 0.7988 | iou 0.7155 | prec 0.8483 | rec 0.8341 | spec 0.9694 | acc 0.9372
   best val dice 0.8293  (no gain for 11/20)
                                                                                                    
[Ep 061/100 |  61.0%]  21.7s | ETA   14.8 min | lr 1.19e-03
   train: loss 0.3534 | dice 0.8292 | iou 0.7441
   val  : loss 0.4333 | dice 0.7984 | iou 0.7108 | prec 0.7922 | rec 0.8887 | spec 0.9388 | acc 0.9254
   best val dice 0.8293  (no gain for 12/20)
                                                                                                    
[Ep 062/100 |  62.0%]  22.4s | ETA   14.4 min | lr 1.14e-03
   train: loss 0.3471 | dice 0.8342 | iou 0.7478
   val  : loss 0.3879 | dice 0.8216 | iou 0.7424 | prec 0.8723 | rec 0.8306 | spec 0.9744 | acc 0.9425
   best val dice 0.8293  (no gain for 13/20)
                                                                                                    
[Ep 063/100 |  63.0%]  22.0s | ETA   14.1 min | lr 1.09e-03
   train: loss 0.3298 | dice 0.8392 | iou 0.7544
   val  : loss 0.4159 | dice 0.8090 | iou 0.7279 | prec 0.8510 | rec 0.8280 | spec 0.9688 | acc 0.9372
   best val dice 0.8293  (no gain for 14/20)
                                                                                                    
[Ep 064/100 |  64.0%]  20.6s | ETA   13.7 min | lr 1.04e-03
   train: loss 0.3332 | dice 0.8391 | iou 0.7540
   val  : loss 0.4145 | dice 0.8094 | iou 0.7199 | prec 0.8132 | rec 0.8804 | spec 0.9461 | acc 0.9292
   best val dice 0.8293  (no gain for 15/20)
                                                                                                    
[Ep 065/100 |  65.0%]  21.3s | ETA   13.3 min | lr 9.88e-04
   train: loss 0.3396 | dice 0.8380 | iou 0.7531
   val  : loss 0.3703 | dice 0.8214 | iou 0.7371 | prec 0.8425 | rec 0.8609 | spec 0.9638 | acc 0.9398
   best val dice 0.8293  (no gain for 16/20)
                                                                                                    
[Ep 066/100 |  66.0%]  21.9s | ETA   12.9 min | lr 9.39e-04
   train: loss 0.3189 | dice 0.8457 | iou 0.7648
   val  : loss 0.3735 | dice 0.8298 | iou 0.7507 | prec 0.8750 | rec 0.8434 | spec 0.9736 | acc 0.9435
   best val dice 0.8298  <-- new best, model saved
                                                                                                    
[Ep 067/100 |  67.0%]  22.4s | ETA   12.5 min | lr 8.91e-04
   train: loss 0.3288 | dice 0.8467 | iou 0.7644
   val  : loss 0.3799 | dice 0.8313 | iou 0.7533 | prec 0.8981 | rec 0.8225 | spec 0.9828 | acc 0.9466
   best val dice 0.8313  <-- new best, model saved
                                                                                                    
[Ep 068/100 |  68.0%]  27.3s | ETA   12.2 min | lr 8.43e-04
   train: loss 0.3220 | dice 0.8466 | iou 0.7639
   val  : loss 0.3570 | dice 0.8313 | iou 0.7524 | prec 0.8914 | rec 0.8330 | spec 0.9807 | acc 0.9480
   best val dice 0.8313  <-- new best, model saved
                                                                                                    
[Ep 069/100 |  69.0%]  24.6s | ETA   11.8 min | lr 7.97e-04
   train: loss 0.3096 | dice 0.8480 | iou 0.7674
   val  : loss 0.3691 | dice 0.8281 | iou 0.7494 | prec 0.8659 | rec 0.8585 | spec 0.9710 | acc 0.9450
   best val dice 0.8313  (no gain for 1/20)
                                                                                                    
[Ep 070/100 |  70.0%]  20.8s | ETA   11.4 min | lr 7.51e-04
   train: loss 0.3127 | dice 0.8483 | iou 0.7675
   val  : loss 0.3695 | dice 0.8317 | iou 0.7551 | prec 0.8917 | rec 0.8287 | spec 0.9826 | acc 0.9493
   best val dice 0.8317  <-- new best, model saved
                                                                                                    
[Ep 071/100 |  71.0%]  21.5s | ETA   11.0 min | lr 7.06e-04
   train: loss 0.2999 | dice 0.8508 | iou 0.7700
   val  : loss 0.3983 | dice 0.8267 | iou 0.7511 | prec 0.8956 | rec 0.8251 | spec 0.9825 | acc 0.9475
   best val dice 0.8317  (no gain for 1/20)
                                                                                                    
[Ep 072/100 |  72.0%]  22.1s | ETA   10.6 min | lr 6.62e-04
   train: loss 0.3149 | dice 0.8485 | iou 0.7668
   val  : loss 0.3694 | dice 0.8392 | iou 0.7643 | prec 0.8976 | rec 0.8364 | spec 0.9812 | acc 0.9506
   best val dice 0.8392  <-- new best, model saved
                                                                                                    
[Ep 073/100 |  73.0%]  22.5s | ETA   10.2 min | lr 6.19e-04
   train: loss 0.3082 | dice 0.8505 | iou 0.7705
   val  : loss 0.3865 | dice 0.8242 | iou 0.7413 | prec 0.8439 | rec 0.8635 | spec 0.9620 | acc 0.9405
   best val dice 0.8392  (no gain for 1/20)
                                                                                                    
[Ep 074/100 |  74.0%]  22.2s | ETA    9.9 min | lr 5.77e-04
   train: loss 0.2960 | dice 0.8530 | iou 0.7731
   val  : loss 0.3913 | dice 0.8267 | iou 0.7502 | prec 0.9006 | rec 0.8186 | spec 0.9846 | acc 0.9470
   best val dice 0.8392  (no gain for 2/20)
                                                                                                    
[Ep 075/100 |  75.0%]  22.6s | ETA    9.5 min | lr 5.37e-04
   train: loss 0.2984 | dice 0.8519 | iou 0.7743
   val  : loss 0.3668 | dice 0.8380 | iou 0.7593 | prec 0.8912 | rec 0.8400 | spec 0.9795 | acc 0.9504
   best val dice 0.8392  (no gain for 3/20)
                                                                                                    
[Ep 076/100 |  76.0%]  22.8s | ETA    9.1 min | lr 4.97e-04
   train: loss 0.2853 | dice 0.8615 | iou 0.7823
   val  : loss 0.3558 | dice 0.8423 | iou 0.7678 | prec 0.8927 | rec 0.8466 | spec 0.9810 | acc 0.9521
   best val dice 0.8423  <-- new best, model saved
                                                                                                    
[Ep 077/100 |  77.0%]  22.1s | ETA    8.7 min | lr 4.59e-04
   train: loss 0.2932 | dice 0.8566 | iou 0.7758
   val  : loss 0.4235 | dice 0.8175 | iou 0.7399 | prec 0.9122 | rec 0.8021 | spec 0.9863 | acc 0.9448
   best val dice 0.8423  (no gain for 1/20)
                                                                                                    
[Ep 078/100 |  78.0%]  21.4s | ETA    8.3 min | lr 4.22e-04
   train: loss 0.2946 | dice 0.8573 | iou 0.7797
   val  : loss 0.3775 | dice 0.8308 | iou 0.7562 | prec 0.8896 | rec 0.8371 | spec 0.9799 | acc 0.9483
   best val dice 0.8423  (no gain for 2/20)
                                                                                                    
[Ep 079/100 |  79.0%]  23.7s | ETA    8.0 min | lr 3.86e-04
   train: loss 0.2901 | dice 0.8558 | iou 0.7766
   val  : loss 0.3714 | dice 0.8327 | iou 0.7563 | prec 0.8942 | rec 0.8311 | spec 0.9817 | acc 0.9491
   best val dice 0.8423  (no gain for 3/20)
                                                                                                    
[Ep 080/100 |  80.0%]  22.4s | ETA    7.6 min | lr 3.52e-04
   train: loss 0.2943 | dice 0.8561 | iou 0.7789
   val  : loss 0.3640 | dice 0.8352 | iou 0.7622 | prec 0.8989 | rec 0.8295 | spec 0.9824 | acc 0.9503
   best val dice 0.8423  (no gain for 4/20)
                                                                                                    
[Ep 081/100 |  81.0%]  22.8s | ETA    7.2 min | lr 3.19e-04
   train: loss 0.2894 | dice 0.8583 | iou 0.7811
   val  : loss 0.3511 | dice 0.8437 | iou 0.7695 | prec 0.8960 | rec 0.8509 | spec 0.9801 | acc 0.9502
   best val dice 0.8437  <-- new best, model saved
                                                                                                    
[Ep 082/100 |  82.0%]  22.4s | ETA    6.8 min | lr 2.87e-04
   train: loss 0.2833 | dice 0.8638 | iou 0.7869
   val  : loss 0.3407 | dice 0.8481 | iou 0.7740 | prec 0.9067 | rec 0.8367 | spec 0.9842 | acc 0.9536
   best val dice 0.8481  <-- new best, model saved
                                                                                                    
[Ep 083/100 |  83.0%]  22.3s | ETA    6.4 min | lr 2.57e-04
   train: loss 0.2880 | dice 0.8613 | iou 0.7835
   val  : loss 0.3753 | dice 0.8351 | iou 0.7606 | prec 0.9025 | rec 0.8211 | spec 0.9854 | acc 0.9500
   best val dice 0.8481  (no gain for 1/20)
                                                                                                    
[Ep 084/100 |  84.0%]  20.6s | ETA    6.1 min | lr 2.29e-04
   train: loss 0.2881 | dice 0.8576 | iou 0.7803
   val  : loss 0.3588 | dice 0.8388 | iou 0.7611 | prec 0.8978 | rec 0.8375 | spec 0.9819 | acc 0.9511
   best val dice 0.8481  (no gain for 2/20)
                                                                                                    
[Ep 085/100 |  85.0%]  21.8s | ETA    5.7 min | lr 2.02e-04
   train: loss 0.2810 | dice 0.8617 | iou 0.7844
   val  : loss 0.3622 | dice 0.8392 | iou 0.7642 | prec 0.9025 | rec 0.8378 | spec 0.9834 | acc 0.9506
   best val dice 0.8481  (no gain for 3/20)
                                                                                                    
[Ep 086/100 |  86.0%]  20.7s | ETA    5.3 min | lr 1.77e-04
   train: loss 0.2745 | dice 0.8639 | iou 0.7849
   val  : loss 0.3664 | dice 0.8379 | iou 0.7619 | prec 0.9042 | rec 0.8349 | spec 0.9835 | acc 0.9503
   best val dice 0.8481  (no gain for 4/20)
                                                                                                    
[Ep 087/100 |  87.0%]  20.1s | ETA    4.9 min | lr 1.53e-04
   train: loss 0.2779 | dice 0.8620 | iou 0.7841
   val  : loss 0.3576 | dice 0.8413 | iou 0.7663 | prec 0.9040 | rec 0.8357 | spec 0.9840 | acc 0.9517
   best val dice 0.8481  (no gain for 5/20)
                                                                                                    
[Ep 088/100 |  88.0%]  26.7s | ETA    4.5 min | lr 1.31e-04
   train: loss 0.2838 | dice 0.8608 | iou 0.7814
   val  : loss 0.3480 | dice 0.8448 | iou 0.7712 | prec 0.9019 | rec 0.8418 | spec 0.9827 | acc 0.9527
   best val dice 0.8481  (no gain for 6/20)
                                                                                                    
[Ep 089/100 |  89.0%]  23.7s | ETA    4.2 min | lr 1.10e-04
   train: loss 0.2719 | dice 0.8667 | iou 0.7914
   val  : loss 0.3400 | dice 0.8472 | iou 0.7731 | prec 0.9037 | rec 0.8426 | spec 0.9831 | acc 0.9531
   best val dice 0.8481  (no gain for 7/20)
                                                                                                    
[Ep 090/100 |  90.0%]  21.6s | ETA    3.8 min | lr 9.14e-05
   train: loss 0.2722 | dice 0.8653 | iou 0.7889
   val  : loss 0.3367 | dice 0.8484 | iou 0.7741 | prec 0.8983 | rec 0.8515 | spec 0.9819 | acc 0.9533
   best val dice 0.8484  <-- new best, model saved
                                                                                                    
[Ep 091/100 |  91.0%]  21.5s | ETA    3.4 min | lr 7.44e-05
   train: loss 0.2879 | dice 0.8600 | iou 0.7817
   val  : loss 0.3467 | dice 0.8444 | iou 0.7693 | prec 0.9073 | rec 0.8405 | spec 0.9843 | acc 0.9524
   best val dice 0.8484  (no gain for 1/20)
                                                                                                    
[Ep 092/100 |  92.0%]  21.4s | ETA    3.0 min | lr 5.91e-05
   train: loss 0.2759 | dice 0.8629 | iou 0.7866
   val  : loss 0.3461 | dice 0.8451 | iou 0.7711 | prec 0.9051 | rec 0.8420 | spec 0.9838 | acc 0.9532
   best val dice 0.8484  (no gain for 2/20)
                                                                                                    
[Ep 093/100 |  93.0%]  21.5s | ETA    2.6 min | lr 4.55e-05
   train: loss 0.2682 | dice 0.8655 | iou 0.7895
   val  : loss 0.3442 | dice 0.8457 | iou 0.7720 | prec 0.9045 | rec 0.8431 | spec 0.9836 | acc 0.9534
   best val dice 0.8484  (no gain for 3/20)
                                                                                                    
[Ep 094/100 |  94.0%]  22.1s | ETA    2.3 min | lr 3.38e-05
   train: loss 0.2752 | dice 0.8674 | iou 0.7904
   val  : loss 0.3441 | dice 0.8458 | iou 0.7724 | prec 0.9022 | rec 0.8451 | spec 0.9830 | acc 0.9534
   best val dice 0.8484  (no gain for 4/20)
                                                                                                    
[Ep 095/100 |  95.0%]  21.1s | ETA    1.9 min | lr 2.38e-05
   train: loss 0.2709 | dice 0.8674 | iou 0.7923
   val  : loss 0.3435 | dice 0.8457 | iou 0.7723 | prec 0.9021 | rec 0.8447 | spec 0.9828 | acc 0.9535
   best val dice 0.8484  (no gain for 5/20)
                                                                                                    
[Ep 096/100 |  96.0%]  22.3s | ETA    1.5 min | lr 1.56e-05
   train: loss 0.2756 | dice 0.8628 | iou 0.7862
   val  : loss 0.3434 | dice 0.8458 | iou 0.7726 | prec 0.9000 | rec 0.8467 | spec 0.9822 | acc 0.9536
   best val dice 0.8484  (no gain for 6/20)
                                                                                                    
[Ep 097/100 |  97.0%]  21.4s | ETA    1.1 min | lr 9.21e-06
   train: loss 0.2725 | dice 0.8666 | iou 0.7899
   val  : loss 0.3424 | dice 0.8462 | iou 0.7731 | prec 0.8996 | rec 0.8476 | spec 0.9820 | acc 0.9537
   best val dice 0.8484  (no gain for 7/20)
                                                                                                    
[Ep 098/100 |  98.0%]  21.5s | ETA    0.8 min | lr 4.65e-06
   train: loss 0.2841 | dice 0.8615 | iou 0.7839
   val  : loss 0.3441 | dice 0.8461 | iou 0.7728 | prec 0.9008 | rec 0.8468 | spec 0.9823 | acc 0.9536
   best val dice 0.8484  (no gain for 8/20)
                                                                                                    
[Ep 099/100 |  99.0%]  21.9s | ETA    0.4 min | lr 1.91e-06
   train: loss 0.2740 | dice 0.8625 | iou 0.7861
   val  : loss 0.3439 | dice 0.8460 | iou 0.7727 | prec 0.9013 | rec 0.8463 | spec 0.9825 | acc 0.9535
   best val dice 0.8484  (no gain for 9/20)
                                                                                                    
[Ep 100/100 | 100.0%]  21.8s | ETA    0.0 min | lr 1.00e-06
   train: loss 0.2637 | dice 0.8698 | iou 0.7956
   val  : loss 0.3432 | dice 0.8462 | iou 0.7731 | prec 0.9002 | rec 0.8472 | spec 0.9822 | acc 0.9536
   best val dice 0.8484  (no gain for 10/20)

"""

In [32]:
training_normal_info= """
[Ep 001/100 |   1.0%]  69.9s | ETA  114.5 min | lr 4.08e-04
   train: loss 1.2046 | dice 0.4254 | iou 0.2976
   val  : loss 1.3489 | dice 0.0000 | iou 0.0000 | prec 1.0000 | rec 0.0000 | spec 1.0000 | acc 0.8192
   best val dice 0.0000  <-- new best, model saved
                                                                                                    
[Ep 002/100 |   2.0%]  43.4s | ETA   92.5 min | lr 6.96e-04
   train: loss 0.9892 | dice 0.4810 | iou 0.3486
   val  : loss 1.2842 | dice 0.0000 | iou 0.0000 | prec 1.0000 | rec 0.0000 | spec 1.0000 | acc 0.8192
   best val dice 0.0000  (no gain for 1/40)
                                                                                                    
[Ep 003/100 |   3.0%]  45.0s | ETA   85.1 min | lr 9.84e-04
   train: loss 0.9403 | dice 0.4778 | iou 0.3515
   val  : loss 0.8321 | dice 0.5549 | iou 0.4200 | prec 0.6155 | rec 0.6348 | spec 0.9299 | acc 0.8598
   best val dice 0.5549  <-- new best, model saved
                                                                                                    
[Ep 004/100 |   4.0%]  43.9s | ETA   80.9 min | lr 1.27e-03
   train: loss 0.8950 | dice 0.5067 | iou 0.3788
   val  : loss 1.3509 | dice 0.5097 | iou 0.3734 | prec 0.4059 | rec 0.9027 | spec 0.7121 | acc 0.7319
   best val dice 0.5549  (no gain for 1/40)
                                                                                                    
[Ep 005/100 |   5.0%]  45.8s | ETA   78.4 min | lr 1.56e-03
   train: loss 0.8721 | dice 0.5280 | iou 0.4019
   val  : loss 0.9566 | dice 0.5771 | iou 0.4369 | prec 0.5156 | rec 0.8103 | spec 0.8463 | acc 0.8270
   best val dice 0.5771  <-- new best, model saved
                                                                                                    
[Ep 006/100 |   6.0%]  42.6s | ETA   75.9 min | lr 1.85e-03
   train: loss 0.8065 | dice 0.5703 | iou 0.4445
   val  : loss 0.9013 | dice 0.5571 | iou 0.4333 | prec 0.6219 | rec 0.6527 | spec 0.9202 | acc 0.8544
   best val dice 0.5771  (no gain for 1/40)
                                                                                                    
[Ep 007/100 |   7.0%]  41.0s | ETA   73.4 min | lr 2.14e-03
   train: loss 0.7690 | dice 0.5983 | iou 0.4746
   val  : loss 1.1068 | dice 0.5432 | iou 0.4086 | prec 0.5208 | rec 0.7228 | spec 0.8553 | acc 0.8177
   best val dice 0.5771  (no gain for 2/40)
                                                                                                    
[Ep 008/100 |   8.0%]  41.2s | ETA   71.4 min | lr 2.42e-03
   train: loss 0.7281 | dice 0.6205 | iou 0.4978
   val  : loss 0.9297 | dice 0.6108 | iou 0.4763 | prec 0.5507 | rec 0.8229 | spec 0.8586 | acc 0.8431
   best val dice 0.6108  <-- new best, model saved
                                                                                                    
[Ep 009/100 |   9.0%]  41.2s | ETA   69.7 min | lr 2.71e-03
   train: loss 0.6931 | dice 0.6506 | iou 0.5334
   val  : loss 0.7294 | dice 0.6619 | iou 0.5377 | prec 0.6430 | rec 0.7927 | spec 0.9129 | acc 0.8766
   best val dice 0.6619  <-- new best, model saved
                                                                                                    
[Ep 010/100 |  10.0%]  41.5s | ETA   68.3 min | lr 3.00e-03
   train: loss 0.6608 | dice 0.6703 | iou 0.5546
   val  : loss 1.1051 | dice 0.5892 | iou 0.4592 | prec 0.5256 | rec 0.8152 | spec 0.8335 | acc 0.8226
   best val dice 0.6619  (no gain for 1/40)
                                                                                                    
[Ep 011/100 |  11.0%]  43.7s | ETA   67.3 min | lr 3.00e-03
   train: loss 0.6459 | dice 0.6769 | iou 0.5650
   val  : loss 0.9401 | dice 0.6103 | iou 0.4789 | prec 0.5420 | rec 0.8181 | spec 0.8572 | acc 0.8413
   best val dice 0.6619  (no gain for 2/40)
                                                                                                    
[Ep 012/100 |  12.0%]  43.9s | ETA   66.3 min | lr 3.00e-03
   train: loss 0.6226 | dice 0.6935 | iou 0.5819
   val  : loss 0.6498 | dice 0.7067 | iou 0.6022 | prec 0.7767 | rec 0.7381 | spec 0.9663 | acc 0.9063
   best val dice 0.7067  <-- new best, model saved
                                                                                                    
[Ep 013/100 |  13.0%]  43.8s | ETA   65.5 min | lr 2.99e-03
   train: loss 0.5838 | dice 0.7188 | iou 0.6092
   val  : loss 0.7200 | dice 0.6815 | iou 0.5599 | prec 0.6660 | rec 0.8110 | spec 0.9030 | acc 0.8834
   best val dice 0.7067  (no gain for 1/40)
                                                                                                    
[Ep 014/100 |  14.0%]  41.0s | ETA   64.2 min | lr 2.99e-03
   train: loss 0.5794 | dice 0.7223 | iou 0.6172
   val  : loss 0.5768 | dice 0.7497 | iou 0.6404 | prec 0.7658 | rec 0.8108 | spec 0.9436 | acc 0.9079
   best val dice 0.7497  <-- new best, model saved
                                                                                                    
[Ep 015/100 |  15.0%]  42.1s | ETA   63.3 min | lr 2.98e-03
   train: loss 0.5557 | dice 0.7299 | iou 0.6272
   val  : loss 0.6169 | dice 0.7118 | iou 0.6013 | prec 0.7745 | rec 0.7623 | spec 0.9649 | acc 0.9084
   best val dice 0.7497  (no gain for 1/40)
                                                                                                    
[Ep 016/100 |  16.0%]  42.1s | ETA   62.3 min | lr 2.97e-03
   train: loss 0.5600 | dice 0.7294 | iou 0.6235
   val  : loss 0.5878 | dice 0.7113 | iou 0.6132 | prec 0.8362 | rec 0.7230 | spec 0.9716 | acc 0.9159
   best val dice 0.7497  (no gain for 2/40)
                                                                                                    
[Ep 017/100 |  17.0%]  41.9s | ETA   61.4 min | lr 2.96e-03
   train: loss 0.5389 | dice 0.7460 | iou 0.6446
   val  : loss 0.6010 | dice 0.7366 | iou 0.6241 | prec 0.7400 | rec 0.8039 | spec 0.9383 | acc 0.9040
   best val dice 0.7497  (no gain for 3/40)
                                                                                                    
[Ep 018/100 |  18.0%]  43.5s | ETA   60.5 min | lr 2.94e-03
   train: loss 0.5405 | dice 0.7425 | iou 0.6400
   val  : loss 0.5185 | dice 0.7425 | iou 0.6418 | prec 0.8150 | rec 0.7460 | spec 0.9684 | acc 0.9191
   best val dice 0.7497  (no gain for 4/40)
                                                                                                    
[Ep 019/100 |  19.0%]  43.3s | ETA   59.7 min | lr 2.93e-03
   train: loss 0.5095 | dice 0.7552 | iou 0.6566
   val  : loss 0.5056 | dice 0.7626 | iou 0.6600 | prec 0.8600 | rec 0.7433 | spec 0.9763 | acc 0.9236
   best val dice 0.7626  <-- new best, model saved
                                                                                                    
[Ep 020/100 |  20.0%]  41.7s | ETA   58.8 min | lr 2.91e-03
   train: loss 0.5068 | dice 0.7558 | iou 0.6564
   val  : loss 0.6323 | dice 0.7360 | iou 0.6206 | prec 0.6830 | rec 0.8930 | spec 0.9001 | acc 0.8950
   best val dice 0.7626  (no gain for 1/40)
                                                                                                    
[Ep 021/100 |  21.0%]  42.1s | ETA   57.9 min | lr 2.89e-03
   train: loss 0.4947 | dice 0.7654 | iou 0.6662
   val  : loss 0.4551 | dice 0.7827 | iou 0.6888 | prec 0.8563 | rec 0.7754 | spec 0.9774 | acc 0.9301
   best val dice 0.7827  <-- new best, model saved
                                                                                                    
[Ep 022/100 |  22.0%]  44.4s | ETA   57.3 min | lr 2.87e-03
   train: loss 0.4903 | dice 0.7683 | iou 0.6716
   val  : loss 0.6509 | dice 0.7182 | iou 0.6029 | prec 0.6794 | rec 0.8594 | spec 0.8917 | acc 0.8833
   best val dice 0.7827  (no gain for 1/40)
                                                                                                    
[Ep 023/100 |  23.0%]  42.9s | ETA   56.5 min | lr 2.85e-03
   train: loss 0.4878 | dice 0.7707 | iou 0.6724
   val  : loss 0.6063 | dice 0.6985 | iou 0.6063 | prec 0.8655 | rec 0.6574 | spec 0.9863 | acc 0.9088
   best val dice 0.7827  (no gain for 2/40)
                                                                                                    
[Ep 024/100 |  24.0%]  43.5s | ETA   55.7 min | lr 2.82e-03
   train: loss 0.4677 | dice 0.7791 | iou 0.6856
   val  : loss 0.4568 | dice 0.7888 | iou 0.6926 | prec 0.8601 | rec 0.7765 | spec 0.9762 | acc 0.9281
   best val dice 0.7888  <-- new best, model saved
                                                                                                    
[Ep 025/100 |  25.0%]  44.8s | ETA   55.0 min | lr 2.80e-03
   train: loss 0.4661 | dice 0.7813 | iou 0.6878
   val  : loss 0.5642 | dice 0.7466 | iou 0.6470 | prec 0.7593 | rec 0.8174 | spec 0.9361 | acc 0.9059
   best val dice 0.7888  (no gain for 1/40)
                                                                                                    
[Ep 026/100 |  26.0%]  43.7s | ETA   54.2 min | lr 2.77e-03
   train: loss 0.4730 | dice 0.7762 | iou 0.6807
   val  : loss 0.4469 | dice 0.8082 | iou 0.7178 | prec 0.8578 | rec 0.8305 | spec 0.9693 | acc 0.9311
   best val dice 0.8082  <-- new best, model saved
                                                                                                    
[Ep 027/100 |  27.0%]  41.6s | ETA   53.4 min | lr 2.74e-03
   train: loss 0.4538 | dice 0.7861 | iou 0.6932
   val  : loss 0.4697 | dice 0.7863 | iou 0.6824 | prec 0.7828 | rec 0.8514 | spec 0.9395 | acc 0.9193
   best val dice 0.8082  (no gain for 1/40)
                                                                                                    
[Ep 028/100 |  28.0%]  39.8s | ETA   52.5 min | lr 2.71e-03
   train: loss 0.4452 | dice 0.7849 | iou 0.6908
   val  : loss 0.9766 | dice 0.5615 | iou 0.4812 | prec 0.9440 | rec 0.4992 | spec 0.9955 | acc 0.8849
   best val dice 0.8082  (no gain for 2/40)
                                                                                                    
[Ep 029/100 |  29.0%]  40.5s | ETA   51.7 min | lr 2.68e-03
   train: loss 0.4376 | dice 0.7931 | iou 0.7010
   val  : loss 0.5682 | dice 0.7359 | iou 0.6549 | prec 0.9273 | rec 0.6899 | spec 0.9892 | acc 0.9227
   best val dice 0.8082  (no gain for 3/40)
                                                                                                    
[Ep 030/100 |  30.0%]  40.9s | ETA   50.8 min | lr 2.65e-03
   train: loss 0.4231 | dice 0.7993 | iou 0.7092
   val  : loss 0.4763 | dice 0.7790 | iou 0.6919 | prec 0.8772 | rec 0.7682 | spec 0.9816 | acc 0.9285
   best val dice 0.8082  (no gain for 4/40)
                                                                                                    
[Ep 031/100 |  31.0%]  40.4s | ETA   50.0 min | lr 2.61e-03
   train: loss 0.4508 | dice 0.7889 | iou 0.6971
   val  : loss 0.4831 | dice 0.7923 | iou 0.7025 | prec 0.8750 | rec 0.7906 | spec 0.9750 | acc 0.9319
   best val dice 0.8082  (no gain for 5/40)
                                                                                                    
[Ep 032/100 |  32.0%]  44.1s | ETA   49.3 min | lr 2.58e-03
   train: loss 0.4230 | dice 0.7994 | iou 0.7084
   val  : loss 0.3756 | dice 0.8284 | iou 0.7351 | prec 0.8467 | rec 0.8561 | spec 0.9634 | acc 0.9382
   best val dice 0.8284  <-- new best, model saved
                                                                                                    
[Ep 033/100 |  33.0%]  40.3s | ETA   48.4 min | lr 2.54e-03
   train: loss 0.4143 | dice 0.8047 | iou 0.7152
   val  : loss 0.4447 | dice 0.8026 | iou 0.7078 | prec 0.7999 | rec 0.8698 | spec 0.9469 | acc 0.9267
   best val dice 0.8284  (no gain for 1/40)
                                                                                                    
[Ep 034/100 |  34.0%]  40.8s | ETA   47.6 min | lr 2.50e-03
   train: loss 0.4078 | dice 0.8080 | iou 0.7205
   val  : loss 0.3984 | dice 0.8091 | iou 0.7252 | prec 0.9163 | rec 0.7899 | spec 0.9861 | acc 0.9377
   best val dice 0.8284  (no gain for 2/40)
                                                                                                    
[Ep 035/100 |  35.0%]  39.5s | ETA   46.8 min | lr 2.46e-03
   train: loss 0.4139 | dice 0.8067 | iou 0.7165
   val  : loss 0.4148 | dice 0.8092 | iou 0.7229 | prec 0.8692 | rec 0.8188 | spec 0.9739 | acc 0.9352
   best val dice 0.8284  (no gain for 3/40)
                                                                                                    
[Ep 036/100 |  36.0%]  40.3s | ETA   46.0 min | lr 2.42e-03
   train: loss 0.4156 | dice 0.7986 | iou 0.7090
   val  : loss 0.4807 | dice 0.7850 | iou 0.7052 | prec 0.8668 | rec 0.7955 | spec 0.9769 | acc 0.9309
   best val dice 0.8284  (no gain for 4/40)
                                                                                                    
[Ep 037/100 |  37.0%]  39.5s | ETA   45.2 min | lr 2.38e-03
   train: loss 0.3937 | dice 0.8082 | iou 0.7205
   val  : loss 0.5089 | dice 0.7692 | iou 0.6766 | prec 0.9160 | rec 0.7221 | spec 0.9886 | acc 0.9289
   best val dice 0.8284  (no gain for 5/40)
                                                                                                    
[Ep 038/100 |  38.0%]  41.7s | ETA   44.4 min | lr 2.34e-03
   train: loss 0.3869 | dice 0.8154 | iou 0.7282
   val  : loss 0.4388 | dice 0.8089 | iou 0.7206 | prec 0.8931 | rec 0.7938 | spec 0.9805 | acc 0.9310
   best val dice 0.8284  (no gain for 6/40)
                                                                                                    
[Ep 039/100 |  39.0%]  40.1s | ETA   43.6 min | lr 2.30e-03
   train: loss 0.3935 | dice 0.8141 | iou 0.7265
   val  : loss 0.3957 | dice 0.8070 | iou 0.7207 | prec 0.9019 | rec 0.7802 | spec 0.9824 | acc 0.9384
   best val dice 0.8284  (no gain for 7/40)
                                                                                                    
[Ep 040/100 |  40.0%]  39.4s | ETA   42.8 min | lr 2.25e-03
   train: loss 0.3862 | dice 0.8194 | iou 0.7328
   val  : loss 0.3922 | dice 0.8225 | iou 0.7288 | prec 0.8383 | rec 0.8567 | spec 0.9634 | acc 0.9357
   best val dice 0.8284  (no gain for 8/40)
                                                                                                    
[Ep 041/100 |  41.0%]  42.3s | ETA   42.1 min | lr 2.20e-03
   train: loss 0.3878 | dice 0.8198 | iou 0.7313
   val  : loss 0.5201 | dice 0.7423 | iou 0.6502 | prec 0.9322 | rec 0.6850 | spec 0.9907 | acc 0.9227
   best val dice 0.8284  (no gain for 9/40)
                                                                                                    
[Ep 042/100 |  42.0%]  42.0s | ETA   41.4 min | lr 2.16e-03
   train: loss 0.3749 | dice 0.8232 | iou 0.7367
   val  : loss 0.4034 | dice 0.8162 | iou 0.7317 | prec 0.8262 | rec 0.8750 | spec 0.9525 | acc 0.9308
   best val dice 0.8284  (no gain for 10/40)
                                                                                                    
[Ep 043/100 |  43.0%]  40.7s | ETA   40.6 min | lr 2.11e-03
   train: loss 0.3572 | dice 0.8318 | iou 0.7461
   val  : loss 0.3809 | dice 0.8124 | iou 0.7350 | prec 0.8757 | rec 0.8198 | spec 0.9816 | acc 0.9417
   best val dice 0.8284  (no gain for 11/40)
                                                                                                    
[Ep 044/100 |  44.0%]  40.2s | ETA   39.8 min | lr 2.06e-03
   train: loss 0.3656 | dice 0.8267 | iou 0.7412
   val  : loss 0.4688 | dice 0.7783 | iou 0.6862 | prec 0.9004 | rec 0.7451 | spec 0.9822 | acc 0.9319
   best val dice 0.8284  (no gain for 12/40)
                                                                                                    
[Ep 045/100 |  45.0%]  42.4s | ETA   39.1 min | lr 2.01e-03
   train: loss 0.3915 | dice 0.8170 | iou 0.7314
   val  : loss 0.4599 | dice 0.7745 | iou 0.6886 | prec 0.8845 | rec 0.7702 | spec 0.9805 | acc 0.9328
   best val dice 0.8284  (no gain for 13/40)
                                                                                                    
[Ep 046/100 |  46.0%]  40.8s | ETA   38.4 min | lr 1.96e-03
   train: loss 0.3502 | dice 0.8357 | iou 0.7509
   val  : loss 0.3714 | dice 0.8176 | iou 0.7390 | prec 0.8539 | rec 0.8554 | spec 0.9704 | acc 0.9439
   best val dice 0.8284  (no gain for 14/40)
                                                                                                    
[Ep 047/100 |  47.0%]  42.2s | ETA   37.7 min | lr 1.91e-03
   train: loss 0.3654 | dice 0.8288 | iou 0.7458
   val  : loss 0.3776 | dice 0.8351 | iou 0.7502 | prec 0.8868 | rec 0.8352 | spec 0.9819 | acc 0.9442
   best val dice 0.8351  <-- new best, model saved
                                                                                                    
[Ep 048/100 |  48.0%]  40.9s | ETA   36.9 min | lr 1.86e-03
   train: loss 0.3664 | dice 0.8280 | iou 0.7438
   val  : loss 0.3766 | dice 0.8267 | iou 0.7420 | prec 0.8196 | rec 0.8985 | spec 0.9524 | acc 0.9357
   best val dice 0.8351  (no gain for 1/40)
                                                                                                    
[Ep 049/100 |  49.0%]  39.8s | ETA   36.2 min | lr 1.81e-03
   train: loss 0.3421 | dice 0.8365 | iou 0.7538
   val  : loss 0.3509 | dice 0.8416 | iou 0.7605 | prec 0.8989 | rec 0.8395 | spec 0.9757 | acc 0.9472
   best val dice 0.8416  <-- new best, model saved
                                                                                                    
[Ep 050/100 |  50.0%]  40.2s | ETA   35.4 min | lr 1.76e-03
   train: loss 0.3463 | dice 0.8359 | iou 0.7523
   val  : loss 0.3398 | dice 0.8447 | iou 0.7685 | prec 0.8903 | rec 0.8543 | spec 0.9785 | acc 0.9487
   best val dice 0.8447  <-- new best, model saved
                                                                                                    
[Ep 051/100 |  51.0%]  40.0s | ETA   34.7 min | lr 1.71e-03
   train: loss 0.3417 | dice 0.8376 | iou 0.7561
   val  : loss 0.4152 | dice 0.7929 | iou 0.7078 | prec 0.8371 | rec 0.8474 | spec 0.9533 | acc 0.9339
   best val dice 0.8447  (no gain for 1/40)
                                                                                                    
[Ep 052/100 |  52.0%]  41.0s | ETA   33.9 min | lr 1.66e-03
   train: loss 0.3343 | dice 0.8390 | iou 0.7589
   val  : loss 0.3853 | dice 0.8307 | iou 0.7492 | prec 0.9461 | rec 0.7819 | spec 0.9932 | acc 0.9434
   best val dice 0.8447  (no gain for 2/40)
                                                                                                    
[Ep 053/100 |  53.0%]  41.5s | ETA   33.2 min | lr 1.61e-03
   train: loss 0.3334 | dice 0.8442 | iou 0.7628
   val  : loss 0.3550 | dice 0.8358 | iou 0.7536 | prec 0.9105 | rec 0.8200 | spec 0.9839 | acc 0.9463
   best val dice 0.8447  (no gain for 3/40)
                                                                                                    
[Ep 054/100 |  54.0%]  41.1s | ETA   32.5 min | lr 1.55e-03
   train: loss 0.3224 | dice 0.8455 | iou 0.7672
   val  : loss 0.3739 | dice 0.8156 | iou 0.7377 | prec 0.9398 | rec 0.7696 | spec 0.9922 | acc 0.9427
   best val dice 0.8447  (no gain for 4/40)
                                                                                                    
[Ep 055/100 |  55.0%]  40.5s | ETA   31.8 min | lr 1.50e-03
   train: loss 0.3350 | dice 0.8414 | iou 0.7595
   val  : loss 0.3488 | dice 0.8380 | iou 0.7566 | prec 0.8641 | rec 0.8693 | spec 0.9677 | acc 0.9436
   best val dice 0.8447  (no gain for 5/40)
                                                                                                    
[Ep 056/100 |  56.0%]  38.8s | ETA   31.0 min | lr 1.45e-03
   train: loss 0.3165 | dice 0.8466 | iou 0.7680
   val  : loss 0.3522 | dice 0.8281 | iou 0.7479 | prec 0.9176 | rec 0.8083 | spec 0.9843 | acc 0.9447
   best val dice 0.8447  (no gain for 6/40)
                                                                                                    
[Ep 057/100 |  57.0%]  39.7s | ETA   30.3 min | lr 1.40e-03
   train: loss 0.3149 | dice 0.8485 | iou 0.7684
   val  : loss 0.3596 | dice 0.8437 | iou 0.7620 | prec 0.9065 | rec 0.8327 | spec 0.9798 | acc 0.9466
   best val dice 0.8447  (no gain for 7/40)
                                                                                                    
[Ep 058/100 |  58.0%]  39.8s | ETA   29.5 min | lr 1.34e-03
   train: loss 0.3317 | dice 0.8432 | iou 0.7618
   val  : loss 0.3485 | dice 0.8360 | iou 0.7532 | prec 0.9104 | rec 0.8186 | spec 0.9847 | acc 0.9461
   best val dice 0.8447  (no gain for 8/40)
                                                                                                    
[Ep 059/100 |  59.0%]  39.7s | ETA   28.8 min | lr 1.29e-03
   train: loss 0.3115 | dice 0.8559 | iou 0.7772
   val  : loss 0.3643 | dice 0.8368 | iou 0.7588 | prec 0.9315 | rec 0.8027 | spec 0.9901 | acc 0.9458
   best val dice 0.8447  (no gain for 9/40)
                                                                                                    
[Ep 060/100 |  60.0%]  39.6s | ETA   28.1 min | lr 1.24e-03
   train: loss 0.3124 | dice 0.8505 | iou 0.7719
   val  : loss 0.3526 | dice 0.8370 | iou 0.7609 | prec 0.9218 | rec 0.8171 | spec 0.9866 | acc 0.9479
   best val dice 0.8447  (no gain for 10/40)
                                                                                                    
[Ep 061/100 |  61.0%]  39.2s | ETA   27.3 min | lr 1.19e-03
   train: loss 0.3072 | dice 0.8518 | iou 0.7729
   val  : loss 0.3288 | dice 0.8512 | iou 0.7688 | prec 0.8528 | rec 0.8943 | spec 0.9624 | acc 0.9445
   best val dice 0.8512  <-- new best, model saved
                                                                                                    
[Ep 062/100 |  62.0%]  39.2s | ETA   26.6 min | lr 1.14e-03
   train: loss 0.3081 | dice 0.8529 | iou 0.7730
   val  : loss 0.3324 | dice 0.8503 | iou 0.7754 | prec 0.9163 | rec 0.8398 | spec 0.9841 | acc 0.9488
   best val dice 0.8512  (no gain for 1/40)
                                                                                                    
[Ep 063/100 |  63.0%]  39.3s | ETA   25.9 min | lr 1.09e-03
   train: loss 0.2968 | dice 0.8578 | iou 0.7800
   val  : loss 0.3525 | dice 0.8429 | iou 0.7659 | prec 0.9085 | rec 0.8350 | spec 0.9853 | acc 0.9472
   best val dice 0.8512  (no gain for 2/40)
                                                                                                    
[Ep 064/100 |  64.0%]  39.7s | ETA   25.2 min | lr 1.04e-03
   train: loss 0.2961 | dice 0.8571 | iou 0.7804
   val  : loss 0.3414 | dice 0.8477 | iou 0.7702 | prec 0.9024 | rec 0.8490 | spec 0.9769 | acc 0.9457
   best val dice 0.8512  (no gain for 3/40)
                                                                                                    
[Ep 065/100 |  65.0%]  40.0s | ETA   24.4 min | lr 9.88e-04
   train: loss 0.2818 | dice 0.8639 | iou 0.7880
   val  : loss 0.3166 | dice 0.8555 | iou 0.7821 | prec 0.9248 | rec 0.8373 | spec 0.9879 | acc 0.9509
   best val dice 0.8555  <-- new best, model saved
                                                                                                    
[Ep 066/100 |  66.0%]  43.8s | ETA   23.8 min | lr 9.39e-04
   train: loss 0.2935 | dice 0.8612 | iou 0.7849
   val  : loss 0.3241 | dice 0.8506 | iou 0.7759 | prec 0.9177 | rec 0.8400 | spec 0.9818 | acc 0.9489
   best val dice 0.8555  (no gain for 1/40)
                                                                                                    
[Ep 067/100 |  67.0%]  39.4s | ETA   23.0 min | lr 8.91e-04
   train: loss 0.2946 | dice 0.8615 | iou 0.7855
   val  : loss 0.3097 | dice 0.8567 | iou 0.7842 | prec 0.9112 | rec 0.8537 | spec 0.9819 | acc 0.9532
   best val dice 0.8567  <-- new best, model saved
                                                                                                    
[Ep 068/100 |  68.0%]  40.3s | ETA   22.3 min | lr 8.43e-04
   train: loss 0.2914 | dice 0.8602 | iou 0.7843
   val  : loss 0.3085 | dice 0.8488 | iou 0.7742 | prec 0.9028 | rec 0.8440 | spec 0.9801 | acc 0.9515
   best val dice 0.8567  (no gain for 1/40)
                                                                                                    
[Ep 069/100 |  69.0%]  41.0s | ETA   21.6 min | lr 7.97e-04
   train: loss 0.2754 | dice 0.8680 | iou 0.7931
   val  : loss 0.2945 | dice 0.8613 | iou 0.7862 | prec 0.8857 | rec 0.8866 | spec 0.9734 | acc 0.9525
   best val dice 0.8613  <-- new best, model saved
                                                                                                    
[Ep 070/100 |  70.0%]  38.6s | ETA   20.9 min | lr 7.51e-04
   train: loss 0.2857 | dice 0.8629 | iou 0.7885
   val  : loss 0.3032 | dice 0.8549 | iou 0.7821 | prec 0.8891 | rec 0.8699 | spec 0.9758 | acc 0.9538
   best val dice 0.8613  (no gain for 1/40)
                                                                                                    
[Ep 071/100 |  71.0%]  42.6s | ETA   20.2 min | lr 7.06e-04
   train: loss 0.2722 | dice 0.8723 | iou 0.7991
   val  : loss 0.3086 | dice 0.8612 | iou 0.7897 | prec 0.9165 | rec 0.8617 | spec 0.9830 | acc 0.9523
   best val dice 0.8613  (no gain for 2/40)
                                                                                                    
[Ep 072/100 |  72.0%]  40.3s | ETA   19.5 min | lr 6.62e-04
   train: loss 0.2846 | dice 0.8639 | iou 0.7884
   val  : loss 0.3711 | dice 0.8400 | iou 0.7660 | prec 0.9210 | rec 0.8253 | spec 0.9871 | acc 0.9474
   best val dice 0.8613  (no gain for 3/40)
                                                                                                    
[Ep 073/100 |  73.0%]  39.7s | ETA   18.8 min | lr 6.19e-04
   train: loss 0.2710 | dice 0.8717 | iou 0.7986
   val  : loss 0.3153 | dice 0.8614 | iou 0.7892 | prec 0.8998 | rec 0.8707 | spec 0.9781 | acc 0.9520
   best val dice 0.8614  <-- new best, model saved
                                                                                                    
[Ep 074/100 |  74.0%]  39.2s | ETA   18.1 min | lr 5.77e-04
   train: loss 0.2605 | dice 0.8752 | iou 0.8031
   val  : loss 0.3072 | dice 0.8546 | iou 0.7820 | prec 0.9116 | rec 0.8521 | spec 0.9827 | acc 0.9526
   best val dice 0.8614  (no gain for 1/40)
                                                                                                    
[Ep 075/100 |  75.0%]  38.8s | ETA   17.4 min | lr 5.37e-04
   train: loss 0.2718 | dice 0.8699 | iou 0.7975
   val  : loss 0.3302 | dice 0.8424 | iou 0.7682 | prec 0.9160 | rec 0.8334 | spec 0.9836 | acc 0.9504
   best val dice 0.8614  (no gain for 2/40)
                                                                                                    
[Ep 076/100 |  76.0%]  39.6s | ETA   16.7 min | lr 4.97e-04
   train: loss 0.2595 | dice 0.8749 | iou 0.8024
   val  : loss 0.3046 | dice 0.8607 | iou 0.7908 | prec 0.9161 | rec 0.8561 | spec 0.9837 | acc 0.9565
   best val dice 0.8614  (no gain for 3/40)
                                                                                                    
[Ep 077/100 |  77.0%]  42.0s | ETA   16.0 min | lr 4.59e-04
   train: loss 0.2650 | dice 0.8706 | iou 0.7971
   val  : loss 0.3243 | dice 0.8511 | iou 0.7784 | prec 0.9299 | rec 0.8307 | spec 0.9873 | acc 0.9517
   best val dice 0.8614  (no gain for 4/40)
                                                                                                    
[Ep 078/100 |  78.0%]  39.1s | ETA   15.3 min | lr 4.22e-04
   train: loss 0.2566 | dice 0.8762 | iou 0.8046
   val  : loss 0.3253 | dice 0.8527 | iou 0.7821 | prec 0.9239 | rec 0.8399 | spec 0.9857 | acc 0.9521
   best val dice 0.8614  (no gain for 5/40)
                                                                                                    
[Ep 079/100 |  79.0%]  39.3s | ETA   14.6 min | lr 3.86e-04
   train: loss 0.2493 | dice 0.8791 | iou 0.8079
   val  : loss 0.3379 | dice 0.8458 | iou 0.7740 | prec 0.9263 | rec 0.8298 | spec 0.9862 | acc 0.9525
   best val dice 0.8614  (no gain for 6/40)
                                                                                                    
[Ep 080/100 |  80.0%]  39.0s | ETA   13.9 min | lr 3.52e-04
   train: loss 0.2508 | dice 0.8799 | iou 0.8086
   val  : loss 0.3114 | dice 0.8550 | iou 0.7854 | prec 0.9170 | rec 0.8512 | spec 0.9821 | acc 0.9541
   best val dice 0.8614  (no gain for 7/40)
                                                                                                    
[Ep 081/100 |  81.0%]  39.4s | ETA   13.2 min | lr 3.19e-04
   train: loss 0.2478 | dice 0.8799 | iou 0.8089
   val  : loss 0.3206 | dice 0.8510 | iou 0.7800 | prec 0.9095 | rec 0.8516 | spec 0.9815 | acc 0.9525
   best val dice 0.8614  (no gain for 8/40)
                                                                                                    
[Ep 082/100 |  82.0%]  39.0s | ETA   12.5 min | lr 2.87e-04
   train: loss 0.2562 | dice 0.8776 | iou 0.8054
   val  : loss 0.3269 | dice 0.8507 | iou 0.7801 | prec 0.9179 | rec 0.8430 | spec 0.9844 | acc 0.9509
   best val dice 0.8614  (no gain for 9/40)
                                                                                                    
[Ep 083/100 |  83.0%]  40.2s | ETA   11.8 min | lr 2.57e-04
   train: loss 0.2530 | dice 0.8777 | iou 0.8071
   val  : loss 0.3004 | dice 0.8586 | iou 0.7892 | prec 0.9148 | rec 0.8520 | spec 0.9839 | acc 0.9531
   best val dice 0.8614  (no gain for 10/40)
                                                                                                    
[Ep 084/100 |  84.0%]  39.1s | ETA   11.1 min | lr 2.29e-04
   train: loss 0.2531 | dice 0.8794 | iou 0.8088
   val  : loss 0.3124 | dice 0.8539 | iou 0.7844 | prec 0.9159 | rec 0.8472 | spec 0.9840 | acc 0.9527
   best val dice 0.8614  (no gain for 11/40)
                                                                                                    
[Ep 085/100 |  85.0%]  39.2s | ETA   10.4 min | lr 2.02e-04
   train: loss 0.2560 | dice 0.8742 | iou 0.8024
   val  : loss 0.3217 | dice 0.8491 | iou 0.7788 | prec 0.9176 | rec 0.8426 | spec 0.9839 | acc 0.9519
   best val dice 0.8614  (no gain for 12/40)
                                                                                                    
[Ep 086/100 |  86.0%]  41.5s | ETA    9.7 min | lr 1.77e-04
   train: loss 0.2466 | dice 0.8823 | iou 0.8123
   val  : loss 0.3194 | dice 0.8506 | iou 0.7803 | prec 0.9185 | rec 0.8440 | spec 0.9841 | acc 0.9524
   best val dice 0.8614  (no gain for 13/40)
                                                                                                    
[Ep 087/100 |  87.0%]  39.9s | ETA    9.0 min | lr 1.53e-04
   train: loss 0.2397 | dice 0.8849 | iou 0.8159
   val  : loss 0.3256 | dice 0.8512 | iou 0.7811 | prec 0.9249 | rec 0.8384 | spec 0.9864 | acc 0.9524
   best val dice 0.8614  (no gain for 14/40)
                                                                                                    
[Ep 088/100 |  88.0%]  38.9s | ETA    8.3 min | lr 1.31e-04
   train: loss 0.2375 | dice 0.8864 | iou 0.8173
   val  : loss 0.3115 | dice 0.8570 | iou 0.7882 | prec 0.9178 | rec 0.8534 | spec 0.9837 | acc 0.9542
   best val dice 0.8614  (no gain for 15/40)
                                                                                                    
[Ep 089/100 |  89.0%]  37.9s | ETA    7.6 min | lr 1.10e-04
   train: loss 0.2374 | dice 0.8865 | iou 0.8166
   val  : loss 0.3195 | dice 0.8528 | iou 0.7830 | prec 0.9198 | rec 0.8459 | spec 0.9844 | acc 0.9527
   best val dice 0.8614  (no gain for 16/40)
                                                                                                    
[Ep 090/100 |  90.0%]  39.6s | ETA    6.9 min | lr 9.14e-05
   train: loss 0.2423 | dice 0.8837 | iou 0.8139
   val  : loss 0.3183 | dice 0.8523 | iou 0.7827 | prec 0.9191 | rec 0.8466 | spec 0.9838 | acc 0.9530
   best val dice 0.8614  (no gain for 17/40)
                                                                                                    
[Ep 091/100 |  91.0%]  39.1s | ETA    6.2 min | lr 7.44e-05
   train: loss 0.2479 | dice 0.8793 | iou 0.8087
   val  : loss 0.3208 | dice 0.8517 | iou 0.7815 | prec 0.9193 | rec 0.8448 | spec 0.9842 | acc 0.9524
   best val dice 0.8614  (no gain for 18/40)
                                                                                                    
[Ep 092/100 |  92.0%]  39.6s | ETA    5.5 min | lr 5.91e-05
   train: loss 0.2385 | dice 0.8871 | iou 0.8173
   val  : loss 0.3209 | dice 0.8515 | iou 0.7818 | prec 0.9209 | rec 0.8437 | spec 0.9848 | acc 0.9524
   best val dice 0.8614  (no gain for 19/40)
                                                                                                    
[Ep 093/100 |  93.0%]  42.1s | ETA    4.8 min | lr 4.55e-05
   train: loss 0.2411 | dice 0.8838 | iou 0.8135
   val  : loss 0.3199 | dice 0.8514 | iou 0.7812 | prec 0.9216 | rec 0.8419 | spec 0.9850 | acc 0.9526
   best val dice 0.8614  (no gain for 20/40)
                                                                                                    
[Ep 094/100 |  94.0%]  41.3s | ETA    4.1 min | lr 3.38e-05
   train: loss 0.2327 | dice 0.8861 | iou 0.8172
   val  : loss 0.3196 | dice 0.8522 | iou 0.7822 | prec 0.9206 | rec 0.8439 | spec 0.9847 | acc 0.9530
   best val dice 0.8614  (no gain for 21/40)
                                                                                                    
[Ep 095/100 |  95.0%]  39.2s | ETA    3.4 min | lr 2.38e-05
   train: loss 0.2427 | dice 0.8818 | iou 0.8125
   val  : loss 0.3135 | dice 0.8542 | iou 0.7841 | prec 0.9173 | rec 0.8486 | spec 0.9835 | acc 0.9531
   best val dice 0.8614  (no gain for 22/40)
                                                                                                    
[Ep 096/100 |  96.0%]  38.9s | ETA    2.8 min | lr 1.56e-05
   train: loss 0.2495 | dice 0.8810 | iou 0.8104
   val  : loss 0.3128 | dice 0.8554 | iou 0.7858 | prec 0.9187 | rec 0.8489 | spec 0.9839 | acc 0.9536
   best val dice 0.8614  (no gain for 23/40)
                                                                                                    
[Ep 097/100 |  97.0%]  42.0s | ETA    2.1 min | lr 9.21e-06
   train: loss 0.2487 | dice 0.8796 | iou 0.8102
   val  : loss 0.3142 | dice 0.8546 | iou 0.7850 | prec 0.9186 | rec 0.8484 | spec 0.9840 | acc 0.9533
   best val dice 0.8614  (no gain for 24/40)
                                                                                                    
[Ep 098/100 |  98.0%]  39.5s | ETA    1.4 min | lr 4.65e-06
   train: loss 0.2391 | dice 0.8835 | iou 0.8128
   val  : loss 0.3141 | dice 0.8544 | iou 0.7848 | prec 0.9194 | rec 0.8473 | spec 0.9842 | acc 0.9533
   best val dice 0.8614  (no gain for 25/40)
                                                                                                    
[Ep 099/100 |  99.0%]  38.8s | ETA    0.7 min | lr 1.91e-06
   train: loss 0.2361 | dice 0.8844 | iou 0.8152
   val  : loss 0.3158 | dice 0.8536 | iou 0.7838 | prec 0.9204 | rec 0.8456 | spec 0.9846 | acc 0.9531
   best val dice 0.8614  (no gain for 26/40)
                                                                                                    
[Ep 100/100 | 100.0%]  39.2s | ETA    0.0 min | lr 1.00e-06
   train: loss 0.2409 | dice 0.8846 | iou 0.8147
   val  : loss 0.3141 | dice 0.8545 | iou 0.7848 | prec 0.9202 | rec 0.8466 | spec 0.9845 | acc 0.9534
   best val dice 0.8614  (no gain for 27/40)
"""

In [34]:
import re
import matplotlib.pyplot as plt

# 1. Training log WITHOUT data compression
training_info_normal =  training_normal_info

# 2. Training log WITH data compression (replace with your compressed log string if you have one)
training_info_compressed = training_compressed_info

# Define the graph title in a variable
GRAPH_TITLE = "Training Time Comparison: Compression vs. No Compression"

# Helper function to extract epochs and times using Regular Expressions
def parse_training_log(log_string):
    epochs = []
    times = []
    pattern = r"\[Ep\s+(\d+)/\d+\s+\|[^\]]+\]\s+([\d\.]+)s"
    matches = re.findall(pattern, log_string)
    for ep, t in matches:
        epochs.append(int(ep))
        times.append(float(t))
    return epochs, times

# Parse both datasets
epochs_normal, times_normal = parse_training_log(training_info_normal)
epochs_comp, times_comp = parse_training_log(training_info_compressed)

# Create the plot
plt.figure(figsize=(12, 6))

# Plot line 1: Without Data Compression
plt.plot(epochs_normal, times_normal, marker='o', linestyle='-', color='tab:blue', 
         linewidth=2, markersize=6, label='Without Data Compression')

# Plot line 2: With Data Compression
plt.plot(epochs_comp, times_comp, marker='s', linestyle='--', color='tab:orange', 
         linewidth=2, markersize=6, label='With Data Compression')

# Set labels and title using the variable
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Time per Epoch (seconds)", fontsize=12)
plt.title(GRAPH_TITLE, fontsize=14, fontweight='bold')

# Customize grid and add a legend
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)

# Prevent tick overlapping (dynamically spaces out labels and tilts them)
all_epochs = sorted(list(set(epochs_normal + epochs_comp)))
tick_step = max(1, len(all_epochs) // 15)
plt.xticks(all_epochs[::tick_step], rotation=45, ha='right')

plt.tight_layout()

# Save the figure
plt.savefig("training_time_comparison2.png", dpi=300)
print("Comparison graph saved successfully as 'training_time_comparison.png'!")

Comparison graph saved successfully as 'training_time_comparison.png'!


In [3]:
"""
Paper figures for the Kvasir-SEG micromodel.

Reads what train() already wrote:
    <BASE>_log.csv        per-epoch metrics
    <BASE>_splits.json    train/val/test file lists
    <BASE>.keras          best model

Training curves (from the CSV, no model needed):
    loss, dice, iou, accuracy, all val metrics, lr, generalization gap, 2x2 overview
Evaluation figures (loads the model):
    train/val/test metric bars, pixel confusion matrix, ROC + PR, per-image Dice
    distribution, qualitative grid (best / median / worst)

pip install tensorflow numpy opencv-python matplotlib
"""
import os, csv, json
import numpy as np
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ============================ CONFIG ================================
# ============================ CONFIG ================================
BASE = r"/mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/SC/tr-elm/micromodel_v3_compressed_train"
OUT_DIR = os.path.join(os.path.dirname(BASE), "figures-datacompress")
IMG_SIZE = 160                 # overwritten automatically from the model in evaluation_figures()
IMG_INTERP = cv2.INTER_AREA    # must match training (v3 uses INTER_AREA)
RUN_EVAL_FIGURES = True        # False: only plot training curves from the CSV
N_ROC_PIXELS = 1_500_000       # pixel subsample for ROC / PR
SEED = 42
# ====================================================================

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
C_TR, C_VA, C_TE, C_GR = "#1f6feb", "#e8590c", "#2f9e44", "#868e96"

plt.rcParams.update({
    "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.titlesize": 11, "legend.frameon": False, "savefig.bbox": "tight",
})
os.makedirs(OUT_DIR, exist_ok=True)


def save(fig, name):
    fig.savefig(os.path.join(OUT_DIR, name + ".png"), dpi=300)
    fig.savefig(os.path.join(OUT_DIR, name + ".pdf"))
    plt.close(fig)
    print("saved", name)


# ---------------------------- TRAINING CURVES -----------------------
def load_log(path):
    with open(path) as f:
        rows = list(csv.DictReader(f))
    cols = {}
    for k in rows[0]:
        cols[k] = np.array([float(r[k]) if r[k] not in ("", "None", "nan") else np.nan for r in rows])
    return cols


def draw_curve(ax, ep, series, ylabel, best_ep, title=None):
    for label, y, color, ls in series:
        ax.plot(ep, y, color=color, ls=ls, lw=1.8, label=label)
    ax.axvline(best_ep, color=C_GR, ls=":", lw=1.2, label=f"best epoch ({int(best_ep)})")
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    if title: ax.set_title(title)
    ax.legend(loc="best")


def single(ep, series, ylabel, best_ep, name, title=None):
    fig, ax = plt.subplots(figsize=(5.5, 3.8))
    draw_curve(ax, ep, series, ylabel, best_ep, title)
    save(fig, name)


def training_curves(log):
    ep = log["epoch"]
    best_ep = ep[np.nanargmax(log["val_dice"])]
    tr_va = lambda t, v: [("Train", log[t], C_TR, "-"), ("Validation", log[v], C_VA, "-")]

    single(ep, tr_va("train_loss", "val_loss"), "Loss (BCE + Dice)", best_ep, "fig_loss", "Training vs validation loss")
    single(ep, tr_va("train_dice", "val_dice"), "Dice coefficient", best_ep, "fig_dice", "Training vs validation Dice")
    single(ep, tr_va("train_iou", "val_iou"), "IoU", best_ep, "fig_iou", "Training vs validation IoU")

    has_tr_acc = "train_accuracy" in log and not np.all(np.isnan(log["train_accuracy"]))
    if has_tr_acc:
        acc_series = tr_va("train_accuracy", "val_accuracy")
        acc_title = "Training vs validation accuracy"
    else:
        acc_series = [("Validation", log["val_accuracy"], C_VA, "-")]
        acc_title = "Validation accuracy (train accuracy not logged)"
    single(ep, acc_series, "Pixel accuracy", best_ep, "fig_accuracy", acc_title)

    # all validation metrics together
    fig, ax = plt.subplots(figsize=(6, 4))
    cmap = plt.get_cmap("tab10")
    for i, k in enumerate(["dice", "iou", "precision", "recall", "specificity", "accuracy"]):
        ax.plot(ep, log[f"val_{k}"], lw=1.6, color=cmap(i), label=k.capitalize())
    ax.set_xlabel("Epoch"); ax.set_ylabel("Score"); ax.set_title("Validation metrics")
    ax.legend(ncol=3, loc="lower right")
    save(fig, "fig_val_metrics")

    # learning rate
    fig, ax = plt.subplots(figsize=(5.5, 3.2))
    ax.plot(ep, log["lr"], color=C_TR, lw=1.8)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Learning rate"); ax.set_title("One-cycle learning-rate schedule")
    save(fig, "fig_lr")

    # generalization gap
    fig, ax = plt.subplots(figsize=(5.5, 3.2))
    ax.plot(ep, log["train_dice"] - log["val_dice"], color=C_GR, lw=1.8)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Train Dice - Val Dice"); ax.set_title("Generalization gap")
    save(fig, "fig_gap")

    # 2x2 overview
    fig, axs = plt.subplots(2, 2, figsize=(10, 7))
    draw_curve(axs[0, 0], ep, tr_va("train_loss", "val_loss"), "Loss", best_ep, "(a) Loss")
    draw_curve(axs[0, 1], ep, tr_va("train_dice", "val_dice"), "Dice", best_ep, "(b) Dice")
    draw_curve(axs[1, 0], ep, tr_va("train_iou", "val_iou"), "IoU", best_ep, "(c) IoU")
    draw_curve(axs[1, 1], ep, acc_series, "Pixel accuracy", best_ep, "(d) Accuracy")
    fig.tight_layout()
    save(fig, "fig_overview")

    print(f"best epoch {int(best_ep)} | val dice {np.nanmax(log['val_dice']):.4f}")


# ---------------------------- EVALUATION ----------------------------
def read_pair(ip, mp):
    img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    m = cv2.resize(m, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    return img, (m > 127).astype(np.uint8)


def per_image_metrics(pred, gt, eps=1e-7):
    p = pred.reshape(len(gt), -1).astype(np.float64); g = gt.reshape(len(gt), -1).astype(np.float64)
    tp = (p * g).sum(1); fp = (p * (1 - g)).sum(1); fn = ((1 - p) * g).sum(1); tn = ((1 - p) * (1 - g)).sum(1)
    return {"dice": (2 * tp + eps) / (2 * tp + fp + fn + eps), "iou": (tp + eps) / (tp + fp + fn + eps),
            "precision": (tp + eps) / (tp + fp + eps), "recall": (tp + eps) / (tp + fn + eps),
            "specificity": (tn + eps) / (tn + fp + eps), "accuracy": (tp + tn) / (tp + tn + fp + fn)}, (tp, fp, fn, tn)


def predict_split(model, pairs, keep=False, chunk=64):
    per = {k: [] for k in ["dice", "iou", "precision", "recall", "specificity", "accuracy"]}
    store = {"probs": [], "gts": [], "imgs": []}
    for s in range(0, len(pairs), chunk):
        data = [read_pair(ip, mp) for ip, mp in pairs[s:s + chunk]]
        imgs = np.stack([d[0] for d in data]); gts = np.stack([d[1] for d in data])
        x = (imgs.astype(np.float32) / 255.0 - MEAN) / STD
        logits = model.predict(x, batch_size=32, verbose=0)[..., 0]
        probs = 1.0 / (1.0 + np.exp(-np.clip(logits, -50, 50)))
        m, _ = per_image_metrics(probs > 0.5, gts)
        for k in per: per[k].append(m[k])
        if keep:
            store["probs"].append(probs.astype(np.float16)); store["gts"].append(gts); store["imgs"].append(imgs)
    per = {k: np.concatenate(v) for k, v in per.items()}
    if keep:
        store = {k: np.concatenate(v) for k, v in store.items()}
    return per, store


def split_bars(res):
    keys = ["dice", "iou", "precision", "recall", "specificity", "accuracy"]
    fig, ax = plt.subplots(figsize=(8, 3.8))
    w = 0.26; x = np.arange(len(keys))
    for i, (name, color) in enumerate([("train", C_TR), ("val", C_VA), ("test", C_TE)]):
        vals = [res[name][k].mean() for k in keys]
        bars = ax.bar(x + (i - 1) * w, vals, w, color=color, label=name.capitalize())
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.005, f"{v:.3f}", ha="center", fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels([k.capitalize() for k in keys])
    ax.set_ylim(0.6, 1.03); ax.set_ylabel("Score"); ax.set_title("Metrics on train / validation / test (best model, no augmentation)")
    ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.1))
    ax.grid(axis="x", visible=False)
    save(fig, "fig_split_metrics")


def confusion_fig(store):
    pred = store["probs"].astype(np.float32) > 0.5
    _, (tp, fp, fn, tn) = per_image_metrics(pred, store["gts"])
    TP, FP, FN, TN = tp.sum(), fp.sum(), fn.sum(), tn.sum()
    cm = np.array([[TN, FP], [FN, TP]]); cmn = cm / cm.sum(1, keepdims=True)
    fig, ax = plt.subplots(figsize=(4.2, 3.8))
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cmn[i, j]*100:.2f}%\n({int(cm[i, j]):,})", ha="center", va="center",
                    color="white" if cmn[i, j] > 0.5 else "black", fontsize=9)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Background", "Polyp"]); ax.set_yticklabels(["Background", "Polyp"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth"); ax.set_title("Pixel-level confusion matrix (test)")
    ax.grid(False)
    fig.colorbar(im, fraction=0.046, pad=0.04)
    save(fig, "fig_confusion")


def roc_pr_fig(store):
    rng = np.random.default_rng(SEED)
    p = store["probs"].astype(np.float32).ravel(); g = store["gts"].ravel().astype(np.float32)
    idx = rng.choice(len(p), min(N_ROC_PIXELS, len(p)), replace=False)
    p, g = p[idx], g[idx]
    o = np.argsort(-p); gs = g[o]
    tps = np.cumsum(gs); fps = np.cumsum(1 - gs)
    tpr = tps / tps[-1]; fpr = fps / fps[-1]; prec = tps / (tps + fps)
    trapz = lambda y, x: float(np.sum(np.diff(x) * (y[1:] + y[:-1]) / 2))
    auc = trapz(tpr, fpr); ap = float(np.sum(np.diff(tpr, prepend=0) * prec))
    fig, axs = plt.subplots(1, 2, figsize=(8.5, 3.8))
    axs[0].plot(fpr, tpr, color=C_TR, lw=2, label=f"AUC = {auc:.4f}")
    axs[0].plot([0, 1], [0, 1], color=C_GR, ls="--", lw=1)
    axs[0].set_xlabel("False positive rate"); axs[0].set_ylabel("True positive rate"); axs[0].set_title("ROC curve"); axs[0].legend(loc="lower right")
    axs[1].plot(tpr, prec, color=C_VA, lw=2, label=f"AP = {ap:.4f}")
    axs[1].set_xlabel("Recall"); axs[1].set_ylabel("Precision"); axs[1].set_title("Precision-Recall curve"); axs[1].legend(loc="lower left")
    axs[1].set_ylim(0, 1.02)
    fig.tight_layout()
    save(fig, "fig_roc_pr")
    print(f"pixel ROC-AUC {auc:.4f} | AP {ap:.4f}")


def dice_dist_fig(test_per):
    d = test_per["dice"]
    fig, ax = plt.subplots(figsize=(5.5, 3.6))
    ax.hist(d, bins=25, color=C_TE, alpha=0.85, edgecolor="white")
    ax.axvline(d.mean(), color="k", ls="--", lw=1.2, label=f"mean {d.mean():.3f}")
    ax.axvline(np.median(d), color=C_VA, ls="-.", lw=1.2, label=f"median {np.median(d):.3f}")
    ax.set_xlabel("Per-image Dice"); ax.set_ylabel("Number of test images"); ax.set_title("Per-image Dice distribution (test)")
    ax.legend(loc="upper left")
    save(fig, "fig_dice_dist")


def qualitative_fig(store, test_per, k=2):
    d = test_per["dice"]; order = np.argsort(d); mid = len(order) // 2
    groups = [("Best", order[::-1][:k]), ("Median", order[mid - k // 2: mid - k // 2 + k]), ("Worst", order[:k])]
    rows = [(g, i) for g, idxs in groups for i in idxs]
    fig, axs = plt.subplots(len(rows), 4, figsize=(8.4, 2.1 * len(rows)))
    for r, (g, i) in enumerate(rows):
        img = store["imgs"][i]; gt = store["gts"][i].astype(bool); pr = store["probs"][i].astype(np.float32) > 0.5
        err = np.zeros((*gt.shape, 3), dtype=np.float32)
        err[gt & pr] = (0.18, 0.62, 0.27); err[~gt & pr] = (0.85, 0.15, 0.15); err[gt & ~pr] = (0.12, 0.40, 0.90)
        for c, (im, cm) in enumerate([(img, None), (gt, "gray"), (pr, "gray"), (err, None)]):
            ax = axs[r, c]; ax.imshow(im, cmap=cm); ax.axis("off")
            if r == 0: ax.set_title(["Image", "Ground truth", "Prediction", "Error map"][c], fontsize=10)
        axs[r, 0].text(-0.04, 0.5, f"{g}\nDice {d[i]:.3f}", transform=axs[r, 0].transAxes, ha="right", va="center", fontsize=9)
    fig.text(0.5, 0.005, "Error map: green = TP, red = FP, blue = FN", ha="center", fontsize=9)
    fig.tight_layout(rect=(0.04, 0.015, 1, 1))
    save(fig, "fig_qualitative")


def evaluation_figures():
    import tensorflow as tf
    model = tf.keras.models.load_model(BASE + ".keras", compile=False)
    with open(BASE + "_splits.json") as f:
        sp = json.load(f)
    res = {}
    for name in ["train", "val", "test"]:
        print("evaluating", name, len(sp[name]))
        res[name], store = predict_split(model, sp[name], keep=(name == "test"))
        if name == "test":
            test_store = store
    split_bars(res)
    confusion_fig(test_store)
    roc_pr_fig(test_store)
    dice_dist_fig(res["test"])
    qualitative_fig(test_store, res["test"])
    print("\nTest (mean over images):", {k: round(float(v.mean()), 4) for k, v in res["test"].items()})


if __name__ == "__main__":
    training_curves(load_log(BASE + "_log.csv"))
    if RUN_EVAL_FIGURES:
        evaluation_figures()
    print("done ->", OUT_DIR)

saved fig_loss
saved fig_dice
saved fig_iou
saved fig_accuracy
saved fig_val_metrics
saved fig_lr
saved fig_gap
saved fig_overview
best epoch 90 | val dice 0.8484
evaluating train 800


I0000 00:00:1792648009.198581    5394 service.cc:153] XLA service 0x7b319c0c1fd0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1792648009.198624    5394 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9 (Driver: 12.7.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.21.1)
I0000 00:00:1792648009.310926    5394 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1792648009.818938    5394 cuda_dnn.cc:461] Loaded cuDNN version 92101
I0000 00:00:1792648032.065110    5394 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


evaluating val 100
evaluating test 100
saved fig_split_metrics
saved fig_confusion
saved fig_roc_pr
pixel ROC-AUC 0.9827 | AP 0.9306
saved fig_dice_dist
saved fig_qualitative

Test (mean over images): {'dice': 0.851, 'iou': 0.7801, 'precision': 0.87, 'recall': 0.8754, 'specificity': 0.9692, 'accuracy': 0.9547}
done -> /mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/SC/tr-elm/figures-datacompress
